#  Statistical Significance Testing

## Main Function 

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

def find_column(df, method, metric):
    """Find the correct column name for a method and metric"""
    method_variations = {
        "SEAL RAG": ["seal_rag", "SEAL_RAG", "seal"],
        "SELF_RAG": ["self_rag", "SELF_RAG", "selfrag"],
        "CRAG_RAG": ["crag_rag", "CRAG_RAG", "crag"],
        "BASIC_RAG": ["basic_rag", "BASIC_RAG", "basic"]
    }
    
    for col in df.columns:
        for variation in method_variations.get(method, [method.lower()]):
            # More precise matching: method in column AND column ends with metric
            if variation.lower() in col.lower() and col.lower().endswith(f"_{metric.lower()}"):
                return col
    return None

def print_column_detection(df, base_method="SEAL RAG", comparison_methods=None, metrics=None):
    """Print which columns were detected for each method/metric combination"""
    
    if comparison_methods is None:
        comparison_methods = ["SELF_RAG", "CRAG_RAG", "BASIC_RAG"]
    
    if metrics is None:
        metrics = ["final_answer_correct", "precision", "recall", "f1"]
    
    print("🔍 COLUMN DETECTION RESULTS")
    print("=" * 100)
    
    all_methods = [base_method] + comparison_methods
    
    for method in all_methods:
        print(f"\n📊 {method}:")
        print("-" * 80)
        
        for metric in metrics:
            col = find_column(df, method, metric)
            if col:
                print(f"   ✅ {metric:<20} → {col}")
            else:
                print(f"   ❌ {metric:<20} → NOT FOUND")
    
    print("\n" + "=" * 100)

def compare_binary_metrics(method1_results, method2_results, method1_name, method2_name):
    """Compare binary metrics using McNemar's test"""
    
    # Convert to integers (True=1, False=0)
    a1 = [int(x) for x in method1_results]
    a2 = [int(x) for x in method2_results]
    
    # Calculate accuracies
    acc1 = sum(a1) / len(a1)
    acc2 = sum(a2) / len(a2)
    
    # Create contingency table with margins
    contingency_table = pd.crosstab(
        pd.Series(a1, name=method1_name), 
        pd.Series(a2, name=method2_name),
        margins=True
    )
    
    # Run McNemar's test
    try:
        table_for_test = contingency_table.iloc[:-1, :-1]  # Remove margins
        result = mcnemar(table_for_test, exact=False)
        p_value = result.pvalue
    except ValueError:
        p_value = 1.0
    
    # Extract counts
    if contingency_table.shape[0] > 1 and contingency_table.shape[1] > 1:
        both_wrong = contingency_table.iloc[0, 0]
        method2_wins = contingency_table.iloc[0, 1]
        method1_wins = contingency_table.iloc[1, 0]
        both_right = contingency_table.iloc[1, 1]
    else:
        both_wrong = method2_wins = method1_wins = both_right = 0
    
    return {
        'contingency_table': contingency_table,
        'p_value': p_value,
        'is_significant': p_value < 0.05,
        'method1_accuracy': acc1,
        'method2_accuracy': acc2,
        'method1_wins': method1_wins,
        'method2_wins': method2_wins,
        'both_correct': both_right,
        'both_wrong': both_wrong,
        'test_type': 'McNemar',
        'statistic_name': 'Chi-squared',
        'degrees_freedom': 1
    }

def compare_continuous_metrics(method1_values, method2_values, method1_name, method2_name):
    """Compare continuous metrics using paired t-test"""
    
    # Convert to numpy arrays
    values1 = np.array(method1_values)
    values2 = np.array(method2_values)
    
    # Remove NaN values
    mask = ~(np.isnan(values1) | np.isnan(values2))
    values1_clean = values1[mask]
    values2_clean = values2[mask]
    
    if len(values1_clean) == 0:
        return {'error': 'No valid data after removing NaN values'}
    
    # Calculate statistics
    mean1 = np.mean(values1_clean)
    mean2 = np.mean(values2_clean)
    std1 = np.std(values1_clean, ddof=1)
    std2 = np.std(values2_clean, ddof=1)
    
    # Calculate differences
    differences = values1_clean - values2_clean
    mean_diff = np.mean(differences)
    std_diff = np.std(differences, ddof=1)
    
    # Paired t-test
    t_statistic, p_value = stats.ttest_rel(values1_clean, values2_clean)
    
    # Effect size (Cohen's d for paired samples)
    cohen_d = mean_diff / std_diff if std_diff > 0 else 0
    
    return {
        'p_value': p_value,
        'is_significant': p_value < 0.05,
        'mean1': mean1,
        'mean2': mean2,
        'std1': std1,
        'std2': std2,
        'mean_difference': mean_diff,
        'cohen_d': cohen_d,
        't_statistic': t_statistic,
        'n_samples': len(values1_clean),
        'test_type': 'Paired T-test',
        'statistic_name': 'T-statistic',
        'degrees_freedom': len(values1_clean) - 1
    }

def print_contingency_matrix(table, method1_name, method2_name):
    """Print a nicely formatted 2x2 contingency matrix"""
    
    print(f"\n📋 CONTINGENCY MATRIX: {method1_name} vs {method2_name}")
    print("=" * 60)
    
    # Remove margins if they exist
    if 'All' in table.index:
        display_table = table.iloc[:-1, :-1]
    else:
        display_table = table
    
    print(f"                    {method2_name}")
    print(f"                 Wrong    Right    Total")
    print("-" * 50)
    
    if display_table.shape[0] >= 2 and display_table.shape[1] >= 2:
        both_wrong = display_table.iloc[0, 0]
        m2_wins = display_table.iloc[0, 1]
        m1_wins = display_table.iloc[1, 0]
        both_right = display_table.iloc[1, 1]
        
        row1_total = both_wrong + m2_wins
        row2_total = m1_wins + both_right
        col1_total = both_wrong + m1_wins
        col2_total = m2_wins + both_right
        grand_total = row1_total + row2_total
        
        print(f"{method1_name:<8} Wrong  {both_wrong:5d}    {m2_wins:5d}    {row1_total:5d}")
        print(f"         Right  {m1_wins:5d}    {both_right:5d}    {row2_total:5d}")
        print("-" * 50)
        print(f"         Total  {col1_total:5d}    {col2_total:5d}    {grand_total:5d}")
        
        print(f"\n🔍 INTERPRETATION:")
        print(f"   Both wrong: {both_wrong:3d} questions")
        print(f"   {method2_name} wins: {m2_wins:3d} questions (when {method1_name} fails)")
        print(f"   {method1_name} wins: {m1_wins:3d} questions (when {method2_name} fails)")
        print(f"   Both right: {both_right:3d} questions")
        print(f"   Disagreements: {m1_wins + m2_wins:3d} questions (key for statistical test)")
        
        if m1_wins + m2_wins > 0:
            if m1_wins > m2_wins:
                ratio = m1_wins / max(m2_wins, 1)
                print(f"   Head-to-head: {method1_name} wins {ratio:.1f}x more often")
            elif m2_wins > m1_wins:
                ratio = m2_wins / max(m1_wins, 1)
                print(f"   Head-to-head: {method2_name} wins {ratio:.1f}x more often")
            else:
                print(f"   Head-to-head: Tied in disagreements")
    else:
        print("   Insufficient data for 2x2 matrix")

def print_continuous_results(result, method1_name, method2_name, metric_name):
    """Print results for continuous metric comparison"""
    
    if 'error' in result:
        print(f"❌ ERROR: {result['error']}")
        return
    
    print(f"\n📈 CONTINUOUS METRIC RESULTS:")
    print(f"   {method1_name}: {result['mean1']:.4f} ± {result['std1']:.4f}")
    print(f"   {method2_name}: {result['mean2']:.4f} ± {result['std2']:.4f}")
    print(f"   Mean difference: {result['mean_difference']:+.4f}")
    print(f"   Sample size: {result['n_samples']} questions")
    
    print(f"\n📊 PAIRED T-TEST RESULTS:")
    print(f"   T-statistic: {result['t_statistic']:.4f}")
    print(f"   P-value: {result['p_value']:.6f}")
    print(f"   Effect size (Cohen's d): {result['cohen_d']:.4f}")
    
    # Significance
    if result['is_significant']:
        better_method = method1_name if result['mean1'] > result['mean2'] else method2_name
        print(f"   Result: 🟢 SIGNIFICANT - {better_method} is significantly better")
    else:
        print(f"   Result: 🔴 NOT SIGNIFICANT - No significant difference")
    
    # Effect size interpretation
    abs_cohen_d = abs(result['cohen_d'])
    if abs_cohen_d < 0.2:
        effect_size = "negligible"
    elif abs_cohen_d < 0.5:
        effect_size = "small"
    elif abs_cohen_d < 0.8:
        effect_size = "medium"
    else:
        effect_size = "large"
    
    print(f"   Effect size interpretation: {effect_size}")

def create_summary_table(results_dict, base_method, comparison_methods, metrics):
    """Create a comprehensive summary table as a DataFrame"""
    
    metric_display = {
        "final_answer_correct": "Accuracy",
        "precision": "Precision", 
        "recall": "Recall",
        "f1": "F1 Score"
    }
    
    summary_data = []
    
    for comparison_method in comparison_methods:
        for metric in metrics:
            if comparison_method in results_dict and metric in results_dict[comparison_method]:
                result = results_dict[comparison_method][metric]
                
                if 'error' not in result:
                    # Common fields
                    row = {
                        'Base_Method': base_method,
                        'Comparison_Method': comparison_method,
                        'Metric': metric_display.get(metric, metric),
                        'Test_Type': result.get('test_type', 'Unknown'),
                        'Statistic_Name': result.get('statistic_name', 'Unknown'),
                        'P_Value': result['p_value'],
                        'Is_Significant': result['is_significant'],
                        'Significance_Level': 'p < 0.001' if result['p_value'] < 0.001 else 
                                           'p < 0.01' if result['p_value'] < 0.01 else
                                           'p < 0.05' if result['p_value'] < 0.05 else 'Not Significant'
                    }
                    
                    # Metric-specific fields
                    if metric == "final_answer_correct":
                        row.update({
                            'Base_Performance': f"{result['method1_accuracy']:.1%}",
                            'Comparison_Performance': f"{result['method2_accuracy']:.1%}",
                            'Performance_Difference': f"{result['method2_accuracy'] - result['method1_accuracy']:+.1%}",
                            'Base_Wins': result['method1_wins'],
                            'Comparison_Wins': result['method2_wins'],
                            'Effect_Size': 'N/A (Binary)',
                            'Statistic_Value': f"χ² = {result.get('chi_squared', 'N/A')}"
                        })
                    else:
                        better_method = base_method if result['mean1'] > result['mean2'] else comparison_method
                        row.update({
                            'Base_Performance': f"{result['mean1']:.3f}",
                            'Comparison_Performance': f"{result['mean2']:.3f}",
                            'Performance_Difference': f"{result['mean_difference']:+.3f}",
                            'Base_Wins': 'N/A (Continuous)',
                            'Comparison_Wins': 'N/A (Continuous)',
                            'Effect_Size': f"{result['cohen_d']:.3f}",
                            'Statistic_Value': f"t = {result['t_statistic']:.3f}"
                        })
                    
                    summary_data.append(row)
                else:
                    # Error case
                    row = {
                        'Base_Method': base_method,
                        'Comparison_Method': comparison_method,
                        'Metric': metric_display.get(metric, metric),
                        'Test_Type': 'ERROR',
                        'Statistic_Name': 'ERROR',
                        'P_Value': np.nan,
                        'Is_Significant': False,
                        'Significance_Level': 'ERROR',
                        'Base_Performance': 'ERROR',
                        'Comparison_Performance': 'ERROR',
                        'Performance_Difference': 'ERROR',
                        'Base_Wins': 'ERROR',
                        'Comparison_Wins': 'ERROR',
                        'Effect_Size': 'ERROR',
                        'Statistic_Value': 'ERROR'
                    }
                    summary_data.append(row)
    
    return pd.DataFrame(summary_data)

def run_comprehensive_comparison_with_summary(df, base_method="SEAL RAG", comparison_methods=None, metrics=None, 
                                            print_detailed=True, return_summary_table=True):
    """
    Run comprehensive comparison and return summary table
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing experiment results
    base_method : str
        Base method to compare others against
    comparison_methods : list
        Methods to compare against base
    metrics : list
        Metrics to analyze
    print_detailed : bool
        Whether to print detailed analysis
    return_summary_table : bool
        Whether to return summary table DataFrame
        
    Returns:
    --------
    dict containing:
        - 'detailed_results': Full results dictionary
        - 'summary_table': Summary DataFrame (if return_summary_table=True)
    """
    
    if comparison_methods is None:
        comparison_methods = ["SELF_RAG", "CRAG_RAG", "BASIC_RAG"]
    
    if metrics is None:
        metrics = ["final_answer_correct", "precision", "recall", "f1"]
    
    if print_detailed:
        print("🚀 COMPREHENSIVE RAG METHOD COMPARISON")
        print("=" * 80)
        print(f"📊 BASE METHOD: {base_method}")
        print("=" * 80)
        
        # Column detection
        print_column_detection(df, base_method, comparison_methods, metrics)
    
    # Store all results
    detailed_results = {}
    
    # Run all comparisons
    for comparison_method in comparison_methods:
        detailed_results[comparison_method] = {}
        
        for metric in metrics:
            # Find columns
            base_col = find_column(df, base_method, metric)
            comp_col = find_column(df, comparison_method, metric)
            
            if base_col is None or comp_col is None:
                detailed_results[comparison_method][metric] = {'error': 'Column not found'}
                continue
            
            # Different handling based on metric type
            if metric == "final_answer_correct":
                # Binary metric
                base_values = df[base_col].astype(bool).tolist()
                comp_values = df[comp_col].astype(bool).tolist()
                
                result = compare_binary_metrics(
                    base_values, comp_values, base_method, comparison_method
                )
                
            else:
                # Continuous metric
                base_values = df[base_col].astype(float)
                comp_values = df[comp_col].astype(float)
                
                result = compare_continuous_metrics(
                    base_values, comp_values, base_method, comparison_method
                )
            
            detailed_results[comparison_method][metric] = result
    
    if print_detailed:
        # Print summary table
        metric_display = {
            "final_answer_correct": "Accuracy",
            "precision": "Precision", 
            "recall": "Recall",
            "f1": "F1 Score"
        }
        
        print(f"\n📋 SUMMARY TABLE - P-VALUES")
        print("-" * 80)
        print(f"{'Metric':<20} {'Test Type':<15} {'vs SELF_RAG':<12} {'vs CRAG_RAG':<12} {'vs BASIC_RAG':<12}")
        print("-" * 80)
        
        for metric in metrics:
            test_type = "McNemar's" if metric == "final_answer_correct" else "T-Test"
            row = f"{metric_display[metric]:<20} {test_type:<15}"
            
            for method in comparison_methods:
                if metric in detailed_results[method]:
                    result = detailed_results[method][metric]
                    if 'error' not in result:
                        p_val = result['p_value']
                        if p_val < 0.001:
                            p_str = "< 0.001***"
                        elif p_val < 0.01:
                            p_str = f"{p_val:.3f}**"
                        elif p_val < 0.05:
                            p_str = f"{p_val:.3f}*"
                        else:
                            p_str = f"{p_val:.3f}"
                        row += f"{p_str:<12}"
                    else:
                        row += f"{'ERROR':<12}"
                else:
                    row += f"{'N/A':<12}"
            print(row)
        
        print("-" * 80)
        print("*** p < 0.001 (highly significant)")
        print("**  p < 0.01  (very significant)")
        print("*   p < 0.05  (significant)")
        
        # Detailed results
        for comparison_method in comparison_methods:
            print(f"\n\n🎯 DETAILED ANALYSIS: {base_method} vs {comparison_method}")
            print("=" * 80)
            
            for metric in metrics:
                print(f"\n📊 METRIC: {metric_display.get(metric, metric).upper()}")
                print("-" * 60)
                
                if metric not in detailed_results[comparison_method]:
                    print("❌ ERROR: Analysis not performed")
                    continue
                    
                result = detailed_results[comparison_method][metric]
                
                if 'error' in result:
                    print(f"❌ ERROR: {result['error']}")
                    continue
                
                # Print column info
                base_col = find_column(df, base_method, metric)
                comp_col = find_column(df, comparison_method, metric)
                print(f"📁 COLUMNS USED:")
                print(f"   {base_method}: {base_col}")
                print(f"   {comparison_method}: {comp_col}")
                
                if metric == "final_answer_correct":
                    # Binary metric results
                    print(f"\n🔍 TEST TYPE: McNemar's Test (Binary Data)")
                    print(f"   {base_method}: {result['method1_accuracy']:.1%} accuracy")
                    print(f"   {comparison_method}: {result['method2_accuracy']:.1%} accuracy")
                    print(f"   Difference: {result['method2_accuracy'] - result['method1_accuracy']:+.1%}")
                    print(f"   P-value: {result['p_value']:.6f}")
                    print(f"   Significant: {'🟢 YES' if result['is_significant'] else '🔴 NO'}")
                    
                    print_contingency_matrix(result['contingency_table'], base_method, comparison_method)
                    
                else:
                    # Continuous metric results
                    print(f"\n🔍 TEST TYPE: Paired T-Test (Continuous Data)")
                    print_continuous_results(result, base_method, comparison_method, metric)
    
    # Create summary table
    output = {'detailed_results': detailed_results}
    
    if return_summary_table:
        summary_table = create_summary_table(detailed_results, base_method, comparison_methods, metrics)
        output['summary_table'] = summary_table
        
        if print_detailed:
            print(f"\n\n📊 SUMMARY TABLE (DataFrame)")
            print("=" * 120)
            print(summary_table.to_string(index=False))
    
    return output

# Main function to use
def analyze_rag_experiments_complete(df, base_method="SEAL RAG", comparison_methods=None, 
                                   metrics=None, print_detailed=True):
    """
    Complete analysis function that returns both detailed results and summary table
    
    Usage:
    ------
    results = analyze_rag_experiments_complete(df)
    
    # Access summary table
    summary_df = results['summary_table']
    
    # Access detailed results
    detailed = results['detailed_results']
    """
    
    return run_comprehensive_comparison_with_summary(
        df=df,
        base_method=base_method,
        comparison_methods=comparison_methods,
        metrics=metrics,
        print_detailed=print_detailed,
        return_summary_table=True
    )

# Usage examples:
# results = analyze_rag_experiments_complete(df)
# summary_table = results['summary_table']
# detailed_results = results['detailed_results']

# Save summary table to CSV
# summary_table.to_csv('rag_comparison_summary.csv', index=False)

# Filter significant results only
# significant_results = summary_table[summary_table['Is_Significant'] == True]

## 2WikiMultihopQA Dataset 

In [7]:
import os
os.getcwd()

'/Users/moshelahmy/Documents/personal/thesis/seal_rag_iclr'

### Model 4o | K = 1

In [8]:
df = pd.read_csv("src/files/experiment_comparison/2WikiMultihopQA/seal_rag_k_1_model_4o.csv")
#df = df.iloc[:, 30:]
df
#/Users/moshelahmy/Documents/personal/thesis/seal_rag_iclr/src/files/experiment_comparison/2WikiMultihopQA/seal_rag_k_1_model_4o.csv

,id,inputs,reference_outputs,basic_rag_200_k_1_model_4o_v1-43ebdafd_outputs,crag_rag_200_k_1_model_4o_v11-51fd11f6_outputs,seal_rag_200_k_1_model_4o-3aba8ef8_outputs,self_rag_200_k_1_model_4o_v1-bad290e1_outputs,basic_rag_200_k_1_model_4o_v1-43ebdafd_run,crag_rag_200_k_1_model_4o_v11-51fd11f6_run,seal_rag_200_k_1_model_4o-3aba8ef8_run,...,seal_rag_200_k_1_model_4o-3aba8ef8_recall,self_rag_200_k_1_model_4o_v1-bad290e1_recall,basic_rag_200_k_1_model_4o_v1-43ebdafd_f1,crag_rag_200_k_1_model_4o_v11-51fd11f6_f1,seal_rag_200_k_1_model_4o-3aba8ef8_f1,self_rag_200_k_1_model_4o_v1-bad290e1_f1,basic_rag_200_k_1_model_4o_v1-43ebdafd_final_answer_correct,crag_rag_200_k_1_model_4o_v11-51fd11f6_final_answer_correct,seal_rag_200_k_1_model_4o-3aba8ef8_final_answer_correct,self_rag_200_k_1_model_4o_v1-bad290e1_final_answer_correct
0,002de458-f6e2-4ef7-991f-6c31359ec71c,"{""context"": ""{'title': ['Richard Annesley, 3rd...","{""answer"": ""2 September 1770""}","{""docs_to_use"": [""Richard Annesley, 2nd Earl A...","{""docs_to_use"": [""Annesley, William Richard (1...","{""docs_to_use"": [""Richard Annesley, 2nd Earl A...","{""docs_to_use"": [], ""response"": ""I don't know....","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,1.00,0.0,0.6667,0.0000,1.0000,0.0000,0.0,0.0,1.0,0.0
1,009279eb-aa2f-4e9a-a388-0570b3d0fad7,"{""context"": ""{'title': ['Caspar Babypants', 'B...","{""answer"": ""Liverpool""}","{""docs_to_use"": [""God (John Lennon song): \""Go...","{""docs_to_use"": [""God (John Lennon song): \""Go...","{""docs_to_use"": [""John Lennon: John Winston On...","{""docs_to_use"": [], ""response"": ""Liverpool, En...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.50,0.0,0.6667,0.6667,0.6667,0.0000,0.0,0.0,1.0,1.0
2,01afabb2-0984-4333-bfff-0b3ad45cc593,"{""context"": ""{'title': ['M. Venkataraju', 'Wor...","{""answer"": ""United States""}","{""docs_to_use"": [""World and Time Enough: World...","{""docs_to_use"": [""World and Time Enough is a 1...","{""docs_to_use"": [""World and Time Enough: World...","{""docs_to_use"": [], ""response"": ""I don't know....","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.50,0.0,0.6667,0.0000,0.6667,0.0000,0.0,0.0,0.0,0.0
3,04491c40-8385-4c7c-9d07-665afd6ef52d,"{""context"": ""{'title': ['James Woolley (clockm...","{""answer"": ""Madame Pasca""}","{""docs_to_use"": [""James T. O'Donohoe: James T....","{""docs_to_use"": [""D6. James A. Donohoe (August...","{""docs_to_use"": [""James T. O'Donohoe: James T....","{""docs_to_use"": [""James A. Donohoe: James A. D...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.50,0.5,0.0000,0.0000,0.5000,0.6667,0.0,1.0,1.0,1.0
4,097adb49-d636-4e90-b9cf-ffed265452f6,"{""context"": ""{'title': ['Takayama Tomoteru', '...","{""answer"": ""Galați""}","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.50,0.5,0.6667,0.6667,0.6667,0.6667,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,fc20c2aa-4e58-4517-ad1a-a050f0283457,"{""context"": ""{'title': ['Language Management',...","{""answer"": ""yes""}","{""docs_to_use"": [""Kaufland: Kaufland is a Germ...","{

In [9]:
df.columns.to_list()

['id',
 'inputs',
 'reference_outputs',
 'basic_rag_200_k_1_model_4o_v1-43ebdafd_outputs',
 'crag_rag_200_k_1_model_4o_v11-51fd11f6_outputs',
 'seal_rag_200_k_1_model_4o-3aba8ef8_outputs',
 'self_rag_200_k_1_model_4o_v1-bad290e1_outputs',
 'basic_rag_200_k_1_model_4o_v1-43ebdafd_run',
 'crag_rag_200_k_1_model_4o_v11-51fd11f6_run',
 'seal_rag_200_k_1_model_4o-3aba8ef8_run',
 'self_rag_200_k_1_model_4o_v1-bad290e1_run',
 'basic_rag_200_k_1_model_4o_v1-43ebdafd_status',
 'crag_rag_200_k_1_model_4o_v11-51fd11f6_status',
 'seal_rag_200_k_1_model_4o-3aba8ef8_status',
 'self_rag_200_k_1_model_4o_v1-bad290e1_status',
 'basic_rag_200_k_1_model_4o_v1-43ebdafd_error',
 'crag_rag_200_k_1_model_4o_v11-51fd11f6_error',
 'seal_rag_200_k_1_model_4o-3aba8ef8_error',
 'self_rag_200_k_1_model_4o_v1-bad290e1_error',
 'basic_rag_200_k_1_model_4o_v1-43ebdafd_latency',
 'crag_rag_200_k_1_model_4o_v11-51fd11f6_latency',
 'seal_rag_200_k_1_model_4o-3aba8ef8_latency',
 'self_rag_200_k_1_model_4o_v1-bad290e1_lat

In [10]:
# Run complete analysis
results = analyze_rag_experiments_complete(df)

# Access the summary table
summary_table = results['summary_table']
print(summary_table)

# Access detailed results
detailed_results = results['detailed_results']

# Save summary to CSV
#summary_table.to_csv('rag_comparison_summary.csv', index=False)

# Filter for significant results only
significant_only = summary_table[summary_table['Is_Significant'] == True]
print("\nSignificant results only:")
print(significant_only)

# Get specific comparison
seal_vs_crag_accuracy = summary_table[
    (summary_table['Comparison_Method'] == 'CRAG_RAG') & 
    (summary_table['Metric'] == 'Accuracy')
]

🚀 COMPREHENSIVE RAG METHOD COMPARISON
📊 BASE METHOD: SEAL RAG
🔍 COLUMN DETECTION RESULTS

📊 SEAL RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → seal_rag_200_k_1_model_4o-3aba8ef8_final_answer_correct
   ✅ precision            → seal_rag_200_k_1_model_4o-3aba8ef8_precision
   ✅ recall               → seal_rag_200_k_1_model_4o-3aba8ef8_recall
   ✅ f1                   → seal_rag_200_k_1_model_4o-3aba8ef8_f1

📊 SELF_RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → self_rag_200_k_1_model_4o_v1-bad290e1_final_answer_correct
   ✅ precision            → self_rag_200_k_1_model_4o_v1-bad290e1_precision
   ✅ recall               → self_rag_200_k_1_model_4o_v1-bad290e1_recall
   ✅ f1                   → self_rag_200_k_1_model_4o_v1-bad290e1_f1

📊 CRAG_RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → cra

### Model 4o | k = 3

In [11]:
df = pd.read_csv("src/files/experiment_comparison/2WikiMultihopQA/seal_rag_k_3_model_4o.csv")
#df = df.iloc[:, 30:]
df

,id,inputs,reference_outputs,basic_rag_200_k_3_model_4o_v3-aee57205_outputs,crag_rag_200_k_3_model_4o_v10-cdeb6d45_outputs,seal_rag_200_k_3_model_4o_v6-1a1c278d_outputs,self_rag_200_k_3_model_4o_v6-9b67a3a9_outputs,basic_rag_200_k_3_model_4o_v3-aee57205_run,crag_rag_200_k_3_model_4o_v10-cdeb6d45_run,seal_rag_200_k_3_model_4o_v6-1a1c278d_run,...,seal_rag_200_k_3_model_4o_v6-1a1c278d_final_answer_correct,self_rag_200_k_3_model_4o_v6-9b67a3a9_final_answer_correct,basic_rag_200_k_3_model_4o_v3-aee57205_f1,crag_rag_200_k_3_model_4o_v10-cdeb6d45_f1,seal_rag_200_k_3_model_4o_v6-1a1c278d_f1,self_rag_200_k_3_model_4o_v6-9b67a3a9_f1,basic_rag_200_k_3_model_4o_v3-aee57205_recall,crag_rag_200_k_3_model_4o_v10-cdeb6d45_recall,seal_rag_200_k_3_model_4o_v6-1a1c278d_recall,self_rag_200_k_3_model_4o_v6-9b67a3a9_recall
0,002de458-f6e2-4ef7-991f-6c31359ec71c,"{""context"": ""{'title': ['Richard Annesley, 3rd...","{""answer"": ""2 September 1770""}","{""docs_to_use"": [""Richard Annesley, 2nd Earl A...","{""docs_to_use"": [""On 02 Sep 1770 [his father] ...","{""docs_to_use"": [""Richard Annesley, 2nd Earl A...","{""docs_to_use"": [], ""response"": ""No informatio...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.0,0.0,0.4000,0.0,0.6667,0.0000,0.5,0.0,0.5,0.00
1,009279eb-aa2f-4e9a-a388-0570b3d0fad7,"{""context"": ""{'title': ['Caspar Babypants', 'B...","{""answer"": ""Liverpool""}","{""docs_to_use"": [""God (John Lennon song): \""Go...","{""docs_to_use"": [""John Lennon: John Winston On...","{""docs_to_use"": [""John Lennon: John Winston On...","{""docs_to_use"": [""John Lennon: John Winston On...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,1.0,1.0,0.8000,0.5,0.6667,0.6667,1.0,0.5,0.5,0.50
2,01afabb2-0984-4333-bfff-0b3ad45cc593,"{""context"": ""{'title': ['M. Venkataraju', 'Wor...","{""answer"": ""United States""}","{""docs_to_use"": [""World and Time Enough: World...","{""docs_to_use"": [""World and Time Enough is a 1...","{""docs_to_use"": [""World and Time Enough: World...","{""docs_to_use"": [""World and Time Enough: World...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.0,1.0,0.4000,0.0,0.6667,0.6667,0.5,0.0,0.5,0.50
3,04491c40-8385-4c7c-9d07-665afd6ef52d,"{""context"": ""{'title': ['James Woolley (clockm...","{""answer"": ""Madame Pasca""}","{""docs_to_use"": [""James T. O'Donohoe: James T....","{""docs_to_use"": [""D6. James A. Donohoe (August...","{""docs_to_use"": [""James A. Donohoe: James A. D...","{""docs_to_use"": [""Madame Pasca: Alice Marie An...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,1.0,1.0,0.8000,0.0,1.0000,0.6667,1.0,0.0,1.0,0.50
4,097adb49-d636-4e90-b9cf-ffed265452f6,"{""context"": ""{'title': ['Takayama Tomoteru', '...","{""answer"": ""Galați""}","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,1.0,1.0,0.8000,0.5,0.6667,0.6667,1.0,0.5,0.5,0.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,fc20c2aa-4e58-4517-ad1a-a050f0283457,"{""context"": ""{'title': ['Language Management',...","{""answer"": ""yes""}","{""docs_to_use"": [""Kaufland: Kaufland is a Germ...","{""docs_to_use"": [""OTRAG: OTRAG(

In [12]:
df.columns.to_list()

['id',
 'inputs',
 'reference_outputs',
 'basic_rag_200_k_3_model_4o_v3-aee57205_outputs',
 'crag_rag_200_k_3_model_4o_v10-cdeb6d45_outputs',
 'seal_rag_200_k_3_model_4o_v6-1a1c278d_outputs',
 'self_rag_200_k_3_model_4o_v6-9b67a3a9_outputs',
 'basic_rag_200_k_3_model_4o_v3-aee57205_run',
 'crag_rag_200_k_3_model_4o_v10-cdeb6d45_run',
 'seal_rag_200_k_3_model_4o_v6-1a1c278d_run',
 'self_rag_200_k_3_model_4o_v6-9b67a3a9_run',
 'basic_rag_200_k_3_model_4o_v3-aee57205_status',
 'crag_rag_200_k_3_model_4o_v10-cdeb6d45_status',
 'seal_rag_200_k_3_model_4o_v6-1a1c278d_status',
 'self_rag_200_k_3_model_4o_v6-9b67a3a9_status',
 'basic_rag_200_k_3_model_4o_v3-aee57205_error',
 'crag_rag_200_k_3_model_4o_v10-cdeb6d45_error',
 'seal_rag_200_k_3_model_4o_v6-1a1c278d_error',
 'self_rag_200_k_3_model_4o_v6-9b67a3a9_error',
 'basic_rag_200_k_3_model_4o_v3-aee57205_latency',
 'crag_rag_200_k_3_model_4o_v10-cdeb6d45_latency',
 'seal_rag_200_k_3_model_4o_v6-1a1c278d_latency',
 'self_rag_200_k_3_model_4o_

In [13]:
# Run complete analysis
results = analyze_rag_experiments_complete(df)

# Access the summary table
summary_table = results['summary_table']
print(summary_table)

# Access detailed results
detailed_results = results['detailed_results']

# Save summary to CSV
#summary_table.to_csv('rag_comparison_summary.csv', index=False)

# Filter for significant results only
significant_only = summary_table[summary_table['Is_Significant'] == True]
print("\nSignificant results only:")
print(significant_only)

# Get specific comparison
seal_vs_crag_accuracy = summary_table[
    (summary_table['Comparison_Method'] == 'CRAG_RAG') & 
    (summary_table['Metric'] == 'Accuracy')
]

🚀 COMPREHENSIVE RAG METHOD COMPARISON
📊 BASE METHOD: SEAL RAG
🔍 COLUMN DETECTION RESULTS

📊 SEAL RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → seal_rag_200_k_3_model_4o_v6-1a1c278d_final_answer_correct
   ✅ precision            → seal_rag_200_k_3_model_4o_v6-1a1c278d_precision
   ✅ recall               → seal_rag_200_k_3_model_4o_v6-1a1c278d_recall
   ✅ f1                   → seal_rag_200_k_3_model_4o_v6-1a1c278d_f1

📊 SELF_RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → self_rag_200_k_3_model_4o_v6-9b67a3a9_final_answer_correct
   ✅ precision            → self_rag_200_k_3_model_4o_v6-9b67a3a9_precision
   ✅ recall               → self_rag_200_k_3_model_4o_v6-9b67a3a9_recall
   ✅ f1                   → self_rag_200_k_3_model_4o_v6-9b67a3a9_f1

📊 CRAG_RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_c

### Model 4o | k = 5

In [14]:
df = pd.read_csv("src/files/experiment_comparison/2WikiMultihopQA/seal_rag_k_5_model_4o.csv")
#df = df.iloc[:, 30:]
df

,id,inputs,reference_outputs,basic_rag_200_k_5_model_4o_v11-e2cb6199_outputs,crag_rag_200_k_5_model_4o_v12-37eb0218_outputs,seal_rag_200_k_5_model_4o_v12-ee7046b6_outputs,self_rag_200_k_5_model_4o_v13-7e02b577_outputs,basic_rag_200_k_5_model_4o_v11-e2cb6199_run,crag_rag_200_k_5_model_4o_v12-37eb0218_run,seal_rag_200_k_5_model_4o_v12-ee7046b6_run,...,seal_rag_200_k_5_model_4o_v12-ee7046b6_f1,self_rag_200_k_5_model_4o_v13-7e02b577_f1,basic_rag_200_k_5_model_4o_v11-e2cb6199_precision,crag_rag_200_k_5_model_4o_v12-37eb0218_precision,seal_rag_200_k_5_model_4o_v12-ee7046b6_precision,self_rag_200_k_5_model_4o_v13-7e02b577_precision,basic_rag_200_k_5_model_4o_v11-e2cb6199_final_answer_correct,crag_rag_200_k_5_model_4o_v12-37eb0218_final_answer_correct,seal_rag_200_k_5_model_4o_v12-ee7046b6_final_answer_correct,self_rag_200_k_5_model_4o_v13-7e02b577_final_answer_correct
0,002de458-f6e2-4ef7-991f-6c31359ec71c,"{""context"": ""{'title': ['Richard Annesley, 3rd...","{""answer"": ""2 September 1770""}","{""docs_to_use"": [""Richard Annesley, 2nd Earl A...","{""docs_to_use"": [""On 02 Sep 1770 [his father] ...","{""docs_to_use"": [""Richard Annesley, 2nd Earl A...","{""docs_to_use"": [], ""response"": ""I don't know....","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.6667,0.0000,0.2,0.0000,1.0,0.0,0.0,1.0,0.0,0.0
1,009279eb-aa2f-4e9a-a388-0570b3d0fad7,"{""context"": ""{'title': ['Caspar Babypants', 'B...","{""answer"": ""Liverpool""}","{""docs_to_use"": [""God (John Lennon song): \""Go...","{""docs_to_use"": [""John Lennon: John Winston On...","{""docs_to_use"": [""John Lennon: John Winston On...","{""docs_to_use"": [""John Lennon: John Winston On...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.6667,0.6667,0.4,0.5000,1.0,1.0,1.0,1.0,1.0,1.0
2,01afabb2-0984-4333-bfff-0b3ad45cc593,"{""context"": ""{'title': ['M. Venkataraju', 'Wor...","{""answer"": ""United States""}","{""docs_to_use"": [""World and Time Enough: World...","{""docs_to_use"": [""You're viewing the Frameline...","{""docs_to_use"": [""World and Time Enough: World...","{""docs_to_use"": [""World and Time Enough: World...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.6667,0.6667,0.2,0.0000,1.0,1.0,0.0,1.0,0.0,0.0
3,04491c40-8385-4c7c-9d07-665afd6ef52d,"{""context"": ""{'title': ['James Woolley (clockm...","{""answer"": ""Madame Pasca""}","{""docs_to_use"": [""James T. O'Donohoe: James T....","{""docs_to_use"": [""D6. James A. Donohoe (August...","{""docs_to_use"": [""James A. Donohoe: James A. D...","{""docs_to_use"": [""James A. Donohoe: James A. D...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,1.0000,1.0000,0.4,0.0000,1.0,1.0,1.0,1.0,1.0,1.0
4,097adb49-d636-4e90-b9cf-ffed265452f6,"{""context"": ""{'title': ['Takayama Tomoteru', '...","{""answer"": ""Galați""}","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.6667,0.6667,0.4,0.5000,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,fc20c2aa-4e58-4517-ad1a-a050f0283457,"{""context"": ""{'title': ['Language Management',...","{""answer"": ""yes""}","{""docs_to_use"": [""Kaufland: Ka

In [15]:
df.columns.to_list()


['id',
 'inputs',
 'reference_outputs',
 'basic_rag_200_k_5_model_4o_v11-e2cb6199_outputs',
 'crag_rag_200_k_5_model_4o_v12-37eb0218_outputs',
 'seal_rag_200_k_5_model_4o_v12-ee7046b6_outputs',
 'self_rag_200_k_5_model_4o_v13-7e02b577_outputs',
 'basic_rag_200_k_5_model_4o_v11-e2cb6199_run',
 'crag_rag_200_k_5_model_4o_v12-37eb0218_run',
 'seal_rag_200_k_5_model_4o_v12-ee7046b6_run',
 'self_rag_200_k_5_model_4o_v13-7e02b577_run',
 'basic_rag_200_k_5_model_4o_v11-e2cb6199_status',
 'crag_rag_200_k_5_model_4o_v12-37eb0218_status',
 'seal_rag_200_k_5_model_4o_v12-ee7046b6_status',
 'self_rag_200_k_5_model_4o_v13-7e02b577_status',
 'basic_rag_200_k_5_model_4o_v11-e2cb6199_error',
 'crag_rag_200_k_5_model_4o_v12-37eb0218_error',
 'seal_rag_200_k_5_model_4o_v12-ee7046b6_error',
 'self_rag_200_k_5_model_4o_v13-7e02b577_error',
 'basic_rag_200_k_5_model_4o_v11-e2cb6199_latency',
 'crag_rag_200_k_5_model_4o_v12-37eb0218_latency',
 'seal_rag_200_k_5_model_4o_v12-ee7046b6_latency',
 'self_rag_200

In [16]:
# Run complete analysis
results = analyze_rag_experiments_complete(df)

# Access the summary table
summary_table = results['summary_table']
print(summary_table)

# Access detailed results
detailed_results = results['detailed_results']

# Save summary to CSV
#summary_table.to_csv('rag_comparison_summary.csv', index=False)

# Filter for significant results only
significant_only = summary_table[summary_table['Is_Significant'] == True]
print("\nSignificant results only:")
print(significant_only)

# Get specific comparison
seal_vs_crag_accuracy = summary_table[
    (summary_table['Comparison_Method'] == 'CRAG_RAG') & 
    (summary_table['Metric'] == 'Accuracy')
]

🚀 COMPREHENSIVE RAG METHOD COMPARISON
📊 BASE METHOD: SEAL RAG
🔍 COLUMN DETECTION RESULTS

📊 SEAL RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → seal_rag_200_k_5_model_4o_v12-ee7046b6_final_answer_correct
   ✅ precision            → seal_rag_200_k_5_model_4o_v12-ee7046b6_precision
   ✅ recall               → seal_rag_200_k_5_model_4o_v12-ee7046b6_recall
   ✅ f1                   → seal_rag_200_k_5_model_4o_v12-ee7046b6_f1

📊 SELF_RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → self_rag_200_k_5_model_4o_v13-7e02b577_final_answer_correct
   ✅ precision            → self_rag_200_k_5_model_4o_v13-7e02b577_precision
   ✅ recall               → self_rag_200_k_5_model_4o_v13-7e02b577_recall
   ✅ f1                   → self_rag_200_k_5_model_4o_v13-7e02b577_f1

📊 CRAG_RAG:
--------------------------------------------------------------------------------
   ✅ final_

### Model 4o-mini | k = 1

In [18]:
df = pd.read_csv("src/files/experiment_comparison/2WikiMultihopQA/seal_rag_k_1_model_4o_mini.csv")
#df = df.iloc[:, 30:]
df

,id,inputs,reference_outputs,basic_rag_200_k_1_model_4o_mini_v1-51d127d7_outputs,crag_rag_200_k_1_model_4o_mini_v1-71b745a4_outputs,seal_rag_200_k_1_model_4o_mini_v2-7489ec26_outputs,self_rag_200_k_1_model_4o_mini_v1-91f7e4ea_outputs,basic_rag_200_k_1_model_4o_mini_v1-51d127d7_run,crag_rag_200_k_1_model_4o_mini_v1-71b745a4_run,seal_rag_200_k_1_model_4o_mini_v2-7489ec26_run,...,seal_rag_200_k_1_model_4o_mini_v2-7489ec26_recall,self_rag_200_k_1_model_4o_mini_v1-91f7e4ea_recall,basic_rag_200_k_1_model_4o_mini_v1-51d127d7_f1,crag_rag_200_k_1_model_4o_mini_v1-71b745a4_f1,seal_rag_200_k_1_model_4o_mini_v2-7489ec26_f1,self_rag_200_k_1_model_4o_mini_v1-91f7e4ea_f1,basic_rag_200_k_1_model_4o_mini_v1-51d127d7_precision,crag_rag_200_k_1_model_4o_mini_v1-71b745a4_precision,seal_rag_200_k_1_model_4o_mini_v2-7489ec26_precision,self_rag_200_k_1_model_4o_mini_v1-91f7e4ea_precision
0,002de458-f6e2-4ef7-991f-6c31359ec71c,"{""context"": ""{'title': ['Richard Annesley, 3rd...","{""answer"": ""2 September 1770""}","{""docs_to_use"": [""Richard Annesley, 2nd Earl A...","{""docs_to_use"": [""Annesley, Richard (1745-1824...","{""docs_to_use"": [""Richard Annesley, 2nd Earl A...","{""docs_to_use"": [], ""response"": ""Don't know."",...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.50,0.00,0.6667,0.0000,0.6667,0.0000,1.0,0.0,1.0,0.0
1,009279eb-aa2f-4e9a-a388-0570b3d0fad7,"{""context"": ""{'title': ['Caspar Babypants', 'B...","{""answer"": ""Liverpool""}","{""docs_to_use"": [""God (John Lennon song): \""Go...","{""docs_to_use"": [""John Winston Lennon was born...","{""docs_to_use"": [""God (John Lennon song): \""Go...","{""docs_to_use"": [], ""response"": ""Not mentioned...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.50,0.00,0.6667,0.0000,0.6667,0.0000,1.0,0.0,1.0,0.0
2,01afabb2-0984-4333-bfff-0b3ad45cc593,"{""context"": ""{'title': ['M. Venkataraju', 'Wor...","{""answer"": ""United States""}","{""docs_to_use"": [""World and Time Enough: World...","{""docs_to_use"": [""You're viewing the Frameline...","{""docs_to_use"": [""World and Time Enough: World...","{""docs_to_use"": [""World and Time Enough: World...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.50,0.50,0.6667,0.0000,0.6667,0.6667,1.0,0.0,1.0,1.0
3,04491c40-8385-4c7c-9d07-665afd6ef52d,"{""context"": ""{'title': ['James Woolley (clockm...","{""answer"": ""Madame Pasca""}","{""docs_to_use"": [""James T. O'Donohoe: James T....","{""docs_to_use"": [""D6. James A. Donohoe (August...","{""docs_to_use"": [""James T. O'Donohoe: James T....","{""docs_to_use"": [], ""response"": ""James A. Dono...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.00,0.00,0.0000,0.0000,0.0000,0.0000,0.0,0.0,0.0,0.0
4,097adb49-d636-4e90-b9cf-ffed265452f6,"{""context"": ""{'title': ['Takayama Tomoteru', '...","{""answer"": ""Galați""}","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.50,0.50,0.6667,0.6667,0.6667,0.6667,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,fc20c2aa-4e58-4517-ad1a-a050f0283457,"{""context"": ""{'title': ['Language Management',...","{""answer"": ""yes""}"

In [19]:
df.columns.to_list()


['id',
 'inputs',
 'reference_outputs',
 'basic_rag_200_k_1_model_4o_mini_v1-51d127d7_outputs',
 'crag_rag_200_k_1_model_4o_mini_v1-71b745a4_outputs',
 'seal_rag_200_k_1_model_4o_mini_v2-7489ec26_outputs',
 'self_rag_200_k_1_model_4o_mini_v1-91f7e4ea_outputs',
 'basic_rag_200_k_1_model_4o_mini_v1-51d127d7_run',
 'crag_rag_200_k_1_model_4o_mini_v1-71b745a4_run',
 'seal_rag_200_k_1_model_4o_mini_v2-7489ec26_run',
 'self_rag_200_k_1_model_4o_mini_v1-91f7e4ea_run',
 'basic_rag_200_k_1_model_4o_mini_v1-51d127d7_status',
 'crag_rag_200_k_1_model_4o_mini_v1-71b745a4_status',
 'seal_rag_200_k_1_model_4o_mini_v2-7489ec26_status',
 'self_rag_200_k_1_model_4o_mini_v1-91f7e4ea_status',
 'basic_rag_200_k_1_model_4o_mini_v1-51d127d7_error',
 'crag_rag_200_k_1_model_4o_mini_v1-71b745a4_error',
 'seal_rag_200_k_1_model_4o_mini_v2-7489ec26_error',
 'self_rag_200_k_1_model_4o_mini_v1-91f7e4ea_error',
 'basic_rag_200_k_1_model_4o_mini_v1-51d127d7_latency',
 'crag_rag_200_k_1_model_4o_mini_v1-71b745a4_lat

In [20]:
# Run complete analysis
results = analyze_rag_experiments_complete(df)

# Access the summary table
summary_table = results['summary_table']
print(summary_table)

# Access detailed results
detailed_results = results['detailed_results']

# Save summary to CSV
#summary_table.to_csv('rag_comparison_summary.csv', index=False)

# Filter for significant results only
significant_only = summary_table[summary_table['Is_Significant'] == True]
print("\nSignificant results only:")
print(significant_only)

# Get specific comparison
seal_vs_crag_accuracy = summary_table[
    (summary_table['Comparison_Method'] == 'CRAG_RAG') & 
    (summary_table['Metric'] == 'Accuracy')
]

🚀 COMPREHENSIVE RAG METHOD COMPARISON
📊 BASE METHOD: SEAL RAG
🔍 COLUMN DETECTION RESULTS

📊 SEAL RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → seal_rag_200_k_1_model_4o_mini_v2-7489ec26_final_answer_correct
   ✅ precision            → seal_rag_200_k_1_model_4o_mini_v2-7489ec26_precision
   ✅ recall               → seal_rag_200_k_1_model_4o_mini_v2-7489ec26_recall
   ✅ f1                   → seal_rag_200_k_1_model_4o_mini_v2-7489ec26_f1

📊 SELF_RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → self_rag_200_k_1_model_4o_mini_v1-91f7e4ea_final_answer_correct
   ✅ precision            → self_rag_200_k_1_model_4o_mini_v1-91f7e4ea_precision
   ✅ recall               → self_rag_200_k_1_model_4o_mini_v1-91f7e4ea_recall
   ✅ f1                   → self_rag_200_k_1_model_4o_mini_v1-91f7e4ea_f1

📊 CRAG_RAG:
------------------------------------------------------------

### Model 4o-mini | k = 3

In [21]:
df = pd.read_csv("src/files/experiment_comparison/2WikiMultihopQA/seal_rag_k_3_model_4o_mini.csv")
#df = df.iloc[:, 30:]
df

,id,inputs,reference_outputs,basic_rag_200_k_3_model_4o_mini_v2-047bf519_outputs,crag_rag_200_k_3_model_4o_mini_v3-6de19cf9_outputs,seal_rag_200_k_3_model_4o_mini_v2-5526b439_outputs,self_rag_200_k_3_model_4o_mini_v3-66fa37d3_outputs,basic_rag_200_k_3_model_4o_mini_v2-047bf519_run,crag_rag_200_k_3_model_4o_mini_v3-6de19cf9_run,seal_rag_200_k_3_model_4o_mini_v2-5526b439_run,...,seal_rag_200_k_3_model_4o_mini_v2-5526b439_final_answer_correct,self_rag_200_k_3_model_4o_mini_v3-66fa37d3_final_answer_correct,basic_rag_200_k_3_model_4o_mini_v2-047bf519_precision,crag_rag_200_k_3_model_4o_mini_v3-6de19cf9_precision,seal_rag_200_k_3_model_4o_mini_v2-5526b439_precision,self_rag_200_k_3_model_4o_mini_v3-66fa37d3_precision,basic_rag_200_k_3_model_4o_mini_v2-047bf519_f1,crag_rag_200_k_3_model_4o_mini_v3-6de19cf9_f1,seal_rag_200_k_3_model_4o_mini_v2-5526b439_f1,self_rag_200_k_3_model_4o_mini_v3-66fa37d3_f1
0,002de458-f6e2-4ef7-991f-6c31359ec71c,"{""context"": ""{'title': ['Richard Annesley, 3rd...","{""answer"": ""2 September 1770""}","{""docs_to_use"": [""Richard Annesley, 2nd Earl A...","{""docs_to_use"": [""Annesley, Richard (1745-1824...","{""docs_to_use"": [""Richard Annesley, 2nd Earl A...","{""docs_to_use"": [], ""response"": ""Don't know."",...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,1.0,0.0,0.3333,0.0,1.0,0.0,0.4000,0.0,0.6667,0.0000
1,009279eb-aa2f-4e9a-a388-0570b3d0fad7,"{""context"": ""{'title': ['Caspar Babypants', 'B...","{""answer"": ""Liverpool""}","{""docs_to_use"": [""God (John Lennon song): \""Go...","{""docs_to_use"": [""John Lennon: John Winston On...","{""docs_to_use"": [""God (John Lennon song): \""Go...","{""docs_to_use"": [""John Lennon: John Winston On...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,1.0,1.0,0.6667,0.5,1.0,1.0,0.8000,0.5,0.6667,0.6667
2,01afabb2-0984-4333-bfff-0b3ad45cc593,"{""context"": ""{'title': ['M. Venkataraju', 'Wor...","{""answer"": ""United States""}","{""docs_to_use"": [""World and Time Enough: World...","{""docs_to_use"": [""World and Time Enough is a 1...","{""docs_to_use"": [""World and Time Enough: World...","{""docs_to_use"": [""Ian Barry (director): Ian Ba...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.0,0.0,0.3333,0.0,1.0,0.0,0.4000,0.0,0.6667,0.0000
3,04491c40-8385-4c7c-9d07-665afd6ef52d,"{""context"": ""{'title': ['James Woolley (clockm...","{""answer"": ""Madame Pasca""}","{""docs_to_use"": [""James T. O'Donohoe: James T....","{""docs_to_use"": [""D6. James A. Donohoe (August...","{""docs_to_use"": [""James T. O'Donohoe: James T....","{""docs_to_use"": [], ""response"": ""Madame Pasca""...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,1.0,1.0,0.6667,0.0,0.0,0.0,0.8000,0.0,0.0000,0.0000
4,097adb49-d636-4e90-b9cf-ffed265452f6,"{""context"": ""{'title': ['Takayama Tomoteru', '...","{""answer"": ""Galați""}","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ștefan I. Nenițescu: Ștefan ...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.0,1.0,0.6667,0.5,1.0,1.0,0.8000,0.5,0.6667,0.6667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,fc20c2aa-4e58-4517-ad1a-a050f0283457,"{""context"": ""{'title': ['Language Management',...","{""an

In [22]:
df.columns.to_list()


['id',
 'inputs',
 'reference_outputs',
 'basic_rag_200_k_3_model_4o_mini_v2-047bf519_outputs',
 'crag_rag_200_k_3_model_4o_mini_v3-6de19cf9_outputs',
 'seal_rag_200_k_3_model_4o_mini_v2-5526b439_outputs',
 'self_rag_200_k_3_model_4o_mini_v3-66fa37d3_outputs',
 'basic_rag_200_k_3_model_4o_mini_v2-047bf519_run',
 'crag_rag_200_k_3_model_4o_mini_v3-6de19cf9_run',
 'seal_rag_200_k_3_model_4o_mini_v2-5526b439_run',
 'self_rag_200_k_3_model_4o_mini_v3-66fa37d3_run',
 'basic_rag_200_k_3_model_4o_mini_v2-047bf519_status',
 'crag_rag_200_k_3_model_4o_mini_v3-6de19cf9_status',
 'seal_rag_200_k_3_model_4o_mini_v2-5526b439_status',
 'self_rag_200_k_3_model_4o_mini_v3-66fa37d3_status',
 'basic_rag_200_k_3_model_4o_mini_v2-047bf519_error',
 'crag_rag_200_k_3_model_4o_mini_v3-6de19cf9_error',
 'seal_rag_200_k_3_model_4o_mini_v2-5526b439_error',
 'self_rag_200_k_3_model_4o_mini_v3-66fa37d3_error',
 'basic_rag_200_k_3_model_4o_mini_v2-047bf519_latency',
 'crag_rag_200_k_3_model_4o_mini_v3-6de19cf9_lat

In [23]:
# Run complete analysis
results = analyze_rag_experiments_complete(df)

# Access the summary table
summary_table = results['summary_table']
print(summary_table)

# Access detailed results
detailed_results = results['detailed_results']

# Save summary to CSV
#summary_table.to_csv('rag_comparison_summary.csv', index=False)

# Filter for significant results only
significant_only = summary_table[summary_table['Is_Significant'] == True]
print("\nSignificant results only:")
print(significant_only)

# Get specific comparison
seal_vs_crag_accuracy = summary_table[
    (summary_table['Comparison_Method'] == 'CRAG_RAG') & 
    (summary_table['Metric'] == 'Accuracy')
]

🚀 COMPREHENSIVE RAG METHOD COMPARISON
📊 BASE METHOD: SEAL RAG
🔍 COLUMN DETECTION RESULTS

📊 SEAL RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → seal_rag_200_k_3_model_4o_mini_v2-5526b439_final_answer_correct
   ✅ precision            → seal_rag_200_k_3_model_4o_mini_v2-5526b439_precision
   ✅ recall               → seal_rag_200_k_3_model_4o_mini_v2-5526b439_recall
   ✅ f1                   → seal_rag_200_k_3_model_4o_mini_v2-5526b439_f1

📊 SELF_RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → self_rag_200_k_3_model_4o_mini_v3-66fa37d3_final_answer_correct
   ✅ precision            → self_rag_200_k_3_model_4o_mini_v3-66fa37d3_precision
   ✅ recall               → self_rag_200_k_3_model_4o_mini_v3-66fa37d3_recall
   ✅ f1                   → self_rag_200_k_3_model_4o_mini_v3-66fa37d3_f1

📊 CRAG_RAG:
------------------------------------------------------------

### Model 4o-mini | k = 5

In [24]:
df = pd.read_csv("src/files/experiment_comparison/2WikiMultihopQA/seal_rag_k_5_model_4o_mini.csv")
#df = df.iloc[:, 30:]
df

,id,inputs,reference_outputs,basic_rag_200_k_5_model_4o_mini_v3-04652ca8_outputs,crag_rag_200_k_5_model_4o_mini_v3-e8fe9982_outputs,seal_rag_200_k_5_model_4o_mini_v2-8080c870_outputs,self_rag_200_k_5_model_4o_mini_v3-44e5efc9_outputs,basic_rag_200_k_5_model_4o_mini_v3-04652ca8_run,crag_rag_200_k_5_model_4o_mini_v3-e8fe9982_run,seal_rag_200_k_5_model_4o_mini_v2-8080c870_run,...,seal_rag_200_k_5_model_4o_mini_v2-8080c870_final_answer_correct,self_rag_200_k_5_model_4o_mini_v3-44e5efc9_final_answer_correct,basic_rag_200_k_5_model_4o_mini_v3-04652ca8_recall,crag_rag_200_k_5_model_4o_mini_v3-e8fe9982_recall,seal_rag_200_k_5_model_4o_mini_v2-8080c870_recall,self_rag_200_k_5_model_4o_mini_v3-44e5efc9_recall,basic_rag_200_k_5_model_4o_mini_v3-04652ca8_precision,crag_rag_200_k_5_model_4o_mini_v3-e8fe9982_precision,seal_rag_200_k_5_model_4o_mini_v2-8080c870_precision,self_rag_200_k_5_model_4o_mini_v3-44e5efc9_precision
0,002de458-f6e2-4ef7-991f-6c31359ec71c,"{""context"": ""{'title': ['Richard Annesley, 3rd...","{""answer"": ""2 September 1770""}","{""docs_to_use"": [""Richard Annesley, 2nd Earl A...","{""docs_to_use"": [""Annesley, Richard (1745-1824...","{""docs_to_use"": [""Richard Annesley, 2nd Earl A...","{""docs_to_use"": [], ""response"": ""Don't know."",...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.0,0.0,0.5,0.0,0.50,0.00,0.2,0.0,1.0,0.0
1,009279eb-aa2f-4e9a-a388-0570b3d0fad7,"{""context"": ""{'title': ['Caspar Babypants', 'B...","{""answer"": ""Liverpool""}","{""docs_to_use"": [""God (John Lennon song): \""Go...","{""docs_to_use"": [""John Lennon: John Winston On...","{""docs_to_use"": [""God (John Lennon song): \""Go...","{""docs_to_use"": [""John Lennon: John Winston On...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,1.0,1.0,1.0,0.5,0.50,0.50,0.4,0.5,1.0,1.0
2,01afabb2-0984-4333-bfff-0b3ad45cc593,"{""context"": ""{'title': ['M. Venkataraju', 'Wor...","{""answer"": ""United States""}","{""docs_to_use"": [""World and Time Enough: World...","{""docs_to_use"": [""World and Time Enough is a 1...","{""docs_to_use"": [""World and Time Enough: World...","{""docs_to_use"": [""World and Time Enough: World...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.0,1.0,0.5,0.0,0.50,0.50,0.2,0.0,1.0,0.5
3,04491c40-8385-4c7c-9d07-665afd6ef52d,"{""context"": ""{'title': ['James Woolley (clockm...","{""answer"": ""Madame Pasca""}","{""docs_to_use"": [""James T. O'Donohoe: James T....","{""docs_to_use"": [""D6. James A. Donohoe (August...","{""docs_to_use"": [""James T. O'Donohoe: James T....","{""docs_to_use"": [], ""response"": ""Madame Pasca""...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,1.0,1.0,1.0,0.0,0.00,0.00,0.4,0.0,0.0,0.0
4,097adb49-d636-4e90-b9cf-ffed265452f6,"{""context"": ""{'title': ['Takayama Tomoteru', '...","{""answer"": ""Galați""}","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ștefan I. Nenițescu: Ștefan ...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",...,0.0,1.0,1.0,0.5,0.50,0.50,0.4,0.5,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,fc20c2aa-4e58-4517-ad1a-a050f0283457,"{""context"": ""{'title': ['Language Management',...","{""answer"": ""yes""}","{""docs_to_use"

In [25]:
df.columns.to_list()

['id',
 'inputs',
 'reference_outputs',
 'basic_rag_200_k_5_model_4o_mini_v3-04652ca8_outputs',
 'crag_rag_200_k_5_model_4o_mini_v3-e8fe9982_outputs',
 'seal_rag_200_k_5_model_4o_mini_v2-8080c870_outputs',
 'self_rag_200_k_5_model_4o_mini_v3-44e5efc9_outputs',
 'basic_rag_200_k_5_model_4o_mini_v3-04652ca8_run',
 'crag_rag_200_k_5_model_4o_mini_v3-e8fe9982_run',
 'seal_rag_200_k_5_model_4o_mini_v2-8080c870_run',
 'self_rag_200_k_5_model_4o_mini_v3-44e5efc9_run',
 'basic_rag_200_k_5_model_4o_mini_v3-04652ca8_status',
 'crag_rag_200_k_5_model_4o_mini_v3-e8fe9982_status',
 'seal_rag_200_k_5_model_4o_mini_v2-8080c870_status',
 'self_rag_200_k_5_model_4o_mini_v3-44e5efc9_status',
 'basic_rag_200_k_5_model_4o_mini_v3-04652ca8_error',
 'crag_rag_200_k_5_model_4o_mini_v3-e8fe9982_error',
 'seal_rag_200_k_5_model_4o_mini_v2-8080c870_error',
 'self_rag_200_k_5_model_4o_mini_v3-44e5efc9_error',
 'basic_rag_200_k_5_model_4o_mini_v3-04652ca8_latency',
 'crag_rag_200_k_5_model_4o_mini_v3-e8fe9982_lat

In [26]:
# Run complete analysis
results = analyze_rag_experiments_complete(df)

# Access the summary table
summary_table = results['summary_table']
print(summary_table)

# Access detailed results
detailed_results = results['detailed_results']

# Save summary to CSV
#summary_table.to_csv('rag_comparison_summary.csv', index=False)

# Filter for significant results only
significant_only = summary_table[summary_table['Is_Significant'] == True]
print("\nSignificant results only:")
print(significant_only)

# Get specific comparison
seal_vs_crag_accuracy = summary_table[
    (summary_table['Comparison_Method'] == 'CRAG_RAG') & 
    (summary_table['Metric'] == 'Accuracy')
]

🚀 COMPREHENSIVE RAG METHOD COMPARISON
📊 BASE METHOD: SEAL RAG
🔍 COLUMN DETECTION RESULTS

📊 SEAL RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → seal_rag_200_k_5_model_4o_mini_v2-8080c870_final_answer_correct
   ✅ precision            → seal_rag_200_k_5_model_4o_mini_v2-8080c870_precision
   ✅ recall               → seal_rag_200_k_5_model_4o_mini_v2-8080c870_recall
   ✅ f1                   → seal_rag_200_k_5_model_4o_mini_v2-8080c870_f1

📊 SELF_RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → self_rag_200_k_5_model_4o_mini_v3-44e5efc9_final_answer_correct
   ✅ precision            → self_rag_200_k_5_model_4o_mini_v3-44e5efc9_precision
   ✅ recall               → self_rag_200_k_5_model_4o_mini_v3-44e5efc9_recall
   ✅ f1                   → self_rag_200_k_5_model_4o_mini_v3-44e5efc9_f1

📊 CRAG_RAG:
------------------------------------------------------------

### Model 4o | seal-rag k = 5 | adaptive k 

In [28]:
df = pd.read_csv("src/files/experiment_comparison/2WikiMultihopQA/adaptive_k_vs_seal_rag/seal_rag_4o.csv")
#df = df.iloc[:, 30:]
df



,id,inputs,reference_outputs,adaptive_k_rag_200_k_50_model_4o_with_buffer_v1-2128f8b2_outputs,adaptive_k_rag_200_k_50_model_4o_without_buffer_v1-6a1ad24a_outputs,seal_rag_200_k_5_model_4o_v12-ee7046b6_outputs,adaptive_k_rag_200_k_50_model_4o_with_buffer_v1-2128f8b2_run,adaptive_k_rag_200_k_50_model_4o_without_buffer_v1-6a1ad24a_run,seal_rag_200_k_5_model_4o_v12-ee7046b6_run,adaptive_k_rag_200_k_50_model_4o_with_buffer_v1-2128f8b2_status,...,seal_rag_200_k_5_model_4o_v12-ee7046b6_final_answer_correct,adaptive_k_rag_200_k_50_model_4o_with_buffer_v1-2128f8b2_f1,adaptive_k_rag_200_k_50_model_4o_without_buffer_v1-6a1ad24a_f1,seal_rag_200_k_5_model_4o_v12-ee7046b6_f1,adaptive_k_rag_200_k_50_model_4o_with_buffer_v1-2128f8b2_recall,adaptive_k_rag_200_k_50_model_4o_without_buffer_v1-6a1ad24a_recall,seal_rag_200_k_5_model_4o_v12-ee7046b6_recall,adaptive_k_rag_200_k_50_model_4o_with_buffer_v1-2128f8b2_precision,adaptive_k_rag_200_k_50_model_4o_without_buffer_v1-6a1ad24a_precision,seal_rag_200_k_5_model_4o_v12-ee7046b6_precision
0,002de458-f6e2-4ef7-991f-6c31359ec71c,"{""context"": ""{'title': ['Richard Annesley, 3rd...","{""answer"": ""2 September 1770""}","{""docs_to_use"": [""Richard Annesley, 2nd Earl A...","{""docs_to_use"": [""Richard Annesley, 2nd Earl A...","{""docs_to_use"": [""Richard Annesley, 2nd Earl A...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,...,0.0,0.3636,0.3333,0.6667,1.0,0.5,0.50,0.2222,0.2500,1.0
1,009279eb-aa2f-4e9a-a388-0570b3d0fad7,"{""context"": ""{'title': ['Caspar Babypants', 'B...","{""answer"": ""Liverpool""}","{""docs_to_use"": [""God (John Lennon song): \""Go...","{""docs_to_use"": [""God (John Lennon song): \""Go...","{""docs_to_use"": [""John Lennon: John Winston On...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,...,1.0,0.4444,1.0000,0.6667,1.0,1.0,0.50,0.2857,1.0000,1.0
2,01afabb2-0984-4333-bfff-0b3ad45cc593,"{""context"": ""{'title': ['M. Venkataraju', 'Wor...","{""answer"": ""United States""}","{""docs_to_use"": [""World and Time Enough: World...","{""docs_to_use"": [""World and Time Enough: World...","{""docs_to_use"": [""World and Time Enough: World...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,...,0.0,0.2500,0.6667,0.6667,0.5,0.5,0.50,0.1667,1.0000,1.0
3,04491c40-8385-4c7c-9d07-665afd6ef52d,"{""context"": ""{'title': ['James Woolley (clockm...","{""answer"": ""Madame Pasca""}","{""docs_to_use"": [""James T. O'Donohoe: James T....","{""docs_to_use"": [""James T. O'Donohoe: James T....","{""docs_to_use"": [""James A. Donohoe: James A. D...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,...,1.0,0.4000,0.8000,1.0000,1.0,1.0,1.00,0.2500,0.6667,1.0
4,097adb49-d636-4e90-b9cf-ffed265452f6,"{""context"": ""{'title': ['Takayama Tomoteru', '...","{""answer"": ""Galați""}","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,...,1.0,0.4444,1.0000,0.6667,1.0,1.0,0.50,0.2857,1.0000,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,fc20c2aa-4e58-4517-ad1a-a050f0283457,"{""context"": ""{'title': ['Language Management',...","{""answer"": ""yes""}","{""docs_to_use"": [""Kaufland: Kaufland is a Germ...","{""docs_to_use"": ["

In [29]:
df.columns.to_list()

['id',
 'inputs',
 'reference_outputs',
 'adaptive_k_rag_200_k_50_model_4o_with_buffer_v1-2128f8b2_outputs',
 'adaptive_k_rag_200_k_50_model_4o_without_buffer_v1-6a1ad24a_outputs',
 'seal_rag_200_k_5_model_4o_v12-ee7046b6_outputs',
 'adaptive_k_rag_200_k_50_model_4o_with_buffer_v1-2128f8b2_run',
 'adaptive_k_rag_200_k_50_model_4o_without_buffer_v1-6a1ad24a_run',
 'seal_rag_200_k_5_model_4o_v12-ee7046b6_run',
 'adaptive_k_rag_200_k_50_model_4o_with_buffer_v1-2128f8b2_status',
 'adaptive_k_rag_200_k_50_model_4o_without_buffer_v1-6a1ad24a_status',
 'seal_rag_200_k_5_model_4o_v12-ee7046b6_status',
 'adaptive_k_rag_200_k_50_model_4o_with_buffer_v1-2128f8b2_error',
 'adaptive_k_rag_200_k_50_model_4o_without_buffer_v1-6a1ad24a_error',
 'seal_rag_200_k_5_model_4o_v12-ee7046b6_error',
 'adaptive_k_rag_200_k_50_model_4o_with_buffer_v1-2128f8b2_latency',
 'adaptive_k_rag_200_k_50_model_4o_without_buffer_v1-6a1ad24a_latency',
 'seal_rag_200_k_5_model_4o_v12-ee7046b6_latency',
 'adaptive_k_rag_200_

In [32]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

# --- METRIC COMPARISON FUNCTIONS (REUSED) ---

def compare_binary_metrics(method1_results, method2_results, method1_name, method2_name):
    """Compare binary metrics using McNemar's test"""
    # Convert to integers (True=1, False=0)
    a1 = [int(x) for x in method1_results]
    a2 = [int(x) for x in method2_results]
    
    acc1 = sum(a1) / len(a1)
    acc2 = sum(a2) / len(a2)
    
    # Create contingency table
    contingency_table = pd.crosstab(
        pd.Series(a1, name=method1_name), 
        pd.Series(a2, name=method2_name)
    )
    
    # Run McNemar's test
    try:
        if contingency_table.shape == (2, 2):
            result = mcnemar(contingency_table, exact=False)
            p_value = result.pvalue
        else:
            # Handle edge cases (e.g., perfect agreement)
            p_value = 1.0
    except ValueError:
        p_value = 1.0
    
    # Extract counts for reporting
    if contingency_table.shape == (2, 2):
        # Table layout:
        #          Method2
        #          0    1
        # Method1 0 [0,0] [0,1] (Method2 wins)
        #         1 [1,0] [1,1]
        # (Method1 wins)
        
        # Note: pd.crosstab indices are sorted (0, 1)
        both_wrong = contingency_table.iloc[0, 0]
        method2_wins = contingency_table.iloc[0, 1]
        method1_wins = contingency_table.iloc[1, 0]
        both_right = contingency_table.iloc[1, 1]
    else:
        both_wrong = method2_wins = method1_wins = both_right = 0
    
    return {
        'p_value': p_value,
        'is_significant': p_value < 0.05,
        'method1_accuracy': acc1,
        'method2_accuracy': acc2,
        'method1_wins': method1_wins,
        'method2_wins': method2_wins,
        'diff': acc1 - acc2
    }

def compare_continuous_metrics(method1_values, method2_values, method1_name, method2_name):
    """Compare continuous metrics using paired t-test"""
    # Convert to numpy arrays and handle NaNs
    v1 = np.array(method1_values, dtype=float)
    v2 = np.array(method2_values, dtype=float)
    
    mask = ~(np.isnan(v1) | np.isnan(v2))
    v1_clean = v1[mask]
    v2_clean = v2[mask]
    
    if len(v1_clean) == 0:
        return {'error': 'No valid data'}
    
    mean1 = np.mean(v1_clean)
    mean2 = np.mean(v2_clean)
    
    # Paired t-test
    t_stat, p_value = stats.ttest_rel(v1_clean, v2_clean)
    
    return {
        'p_value': p_value,
        'is_significant': p_value < 0.05,
        'mean1': mean1,
        'mean2': mean2,
        'diff': mean1 - mean2
    }

# --- MAIN ANALYSIS FUNCTION ---

def analyze_seal_vs_adaptive(csv_path):
    print(f"🚀 LOADING DATA from {csv_path}...")
    df = pd.read_csv(csv_path)
    
    # --- CONFIGURATION: Method Prefixes ---
    # Extracted from your provided column list
    methods = {
        "SEAL_RAG": "seal_rag_200_k_5_model_4o_v12-ee7046b6",
        "Adaptive_K (Buffer)": "adaptive_k_rag_200_k_50_model_4o_with_buffer_v1-2128f8b2",
        "Adaptive_K (No Buffer)": "adaptive_k_rag_200_k_50_model_4o_without_buffer_v1-6a1ad24a"
    }
    
    metrics = ["final_answer_correct", "precision", "recall", "f1"]
    
    print("\n📊 COMPARISON RESULTS (Base: SEAL_RAG)")
    print("=" * 100)
    
    # Iterate over comparisons
    base_name = "SEAL_RAG"
    base_prefix = methods[base_name]
    
    comparisons = [
        "Adaptive_K (Buffer)", 
        "Adaptive_K (No Buffer)"
    ]
    
    results_table = []
    
    for comp_name in comparisons:
        comp_prefix = methods[comp_name]
        print(f"\n⚔️  {base_name} vs {comp_name}")
        print("-" * 80)
        
        for metric in metrics:
            # Construct column names
            col_base = f"{base_prefix}_{metric}"
            col_comp = f"{comp_prefix}_{metric}"
            
            # Check if columns exist
            if col_base not in df.columns or col_comp not in df.columns:
                print(f"⚠️  Missing columns for {metric}")
                continue
                
            # Run appropriate test
            if metric == "final_answer_correct":
                # Binary
                res = compare_binary_metrics(
                    df[col_base], df[col_comp], base_name, comp_name
                )
                
                # Print row
                significance = "**" if res['is_significant'] else ""
                print(f"   {metric:<20} | {base_name}: {res['method1_accuracy']:.1%} | {comp_name}: {res['method2_accuracy']:.1%} | Diff: {res['diff']:+.1%} | p={res['p_value']:.4f} {significance}")
                
                # Add to summary list
                results_table.append({
                    "Comparison": f"{base_name} vs {comp_name}",
                    "Metric": "Accuracy",
                    "Base Score": res['method1_accuracy'],
                    "Comp Score": res['method2_accuracy'],
                    "Diff": res['diff'],
                    "P-Value": res['p_value'],
                    "Significant": res['is_significant']
                })
                
            else:
                # Continuous
                res = compare_continuous_metrics(
                    df[col_base], df[col_comp], base_name, comp_name
                )
                
                # Print row
                significance = "**" if res['is_significant'] else ""
                print(f"   {metric:<20} | {base_name}: {res['mean1']:.3f} | {comp_name}: {res['mean2']:.3f} | Diff: {res['diff']:+.3f} | p={res['p_value']:.4f} {significance}")

                # Add to summary list
                results_table.append({
                    "Comparison": f"{base_name} vs {comp_name}",
                    "Metric": metric.capitalize(),
                    "Base Score": res['mean1'],
                    "Comp Score": res['mean2'],
                    "Diff": res['diff'],
                    "P-Value": res['p_value'],
                    "Significant": res['is_significant']
                })

    print("\n" + "=" * 100)
    
    # Return summary DataFrame
    return pd.DataFrame(results_table)

# --- USAGE ---
df_summary = analyze_seal_vs_adaptive("src/files/experiment_comparison/2WikiMultihopQA/adaptive_k_vs_seal_rag/seal_rag_4o.csv")
print("\nSUMMARY DATAFRAME:")
print(df_summary)

🚀 LOADING DATA from src/files/experiment_comparison/2WikiMultihopQA/adaptive_k_vs_seal_rag/seal_rag_4o.csv...

📊 COMPARISON RESULTS (Base: SEAL_RAG)

⚔️  SEAL_RAG vs Adaptive_K (Buffer)
--------------------------------------------------------------------------------
   final_answer_correct | SEAL_RAG: 74.5% | Adaptive_K (Buffer): 66.5% | Diff: +8.0% | p=0.0206 **
   precision            | SEAL_RAG: 0.963 | Adaptive_K (Buffer): 0.257 | Diff: +0.706 | p=0.0000 **
   recall               | SEAL_RAG: 0.772 | Adaptive_K (Buffer): 0.770 | Diff: +0.002 | p=0.9134 
   f1                   | SEAL_RAG: 0.836 | Adaptive_K (Buffer): 0.377 | Diff: +0.459 | p=0.0000 **

⚔️  SEAL_RAG vs Adaptive_K (No Buffer)
--------------------------------------------------------------------------------
   final_answer_correct | SEAL_RAG: 74.5% | Adaptive_K (No Buffer): 41.5% | Diff: +33.0% | p=0.0000 **
   precision            | SEAL_RAG: 0.963 | Adaptive_K (No Buffer): 0.856 | Diff: +0.107 | p=0.0000 **
   recall

### Model 4o-mini | seal-rag k = 5 | adaptive k 

In [36]:
df = pd.read_csv("src/files/experiment_comparison/2WikiMultihopQA/adaptive_k_vs_seal_rag/seal_rag_4o_mini.csv")
#df = df.iloc[:, 30:]
df


,id,inputs,reference_outputs,adaptive_k_rag_200_k_50_model_4o_mini_with_buffer_v3-772915b6_outputs,adaptive_k_rag_200_k_50_model_4o_mini_without_buffer_v1-e49a13e1_outputs,seal_rag_200_k_5_model_4o_mini_v10-390ea694_outputs,adaptive_k_rag_200_k_50_model_4o_mini_with_buffer_v3-772915b6_run,adaptive_k_rag_200_k_50_model_4o_mini_without_buffer_v1-e49a13e1_run,seal_rag_200_k_5_model_4o_mini_v10-390ea694_run,adaptive_k_rag_200_k_50_model_4o_mini_with_buffer_v3-772915b6_status,...,seal_rag_200_k_5_model_4o_mini_v10-390ea694_f1,adaptive_k_rag_200_k_50_model_4o_mini_with_buffer_v3-772915b6_recall,adaptive_k_rag_200_k_50_model_4o_mini_without_buffer_v1-e49a13e1_recall,seal_rag_200_k_5_model_4o_mini_v10-390ea694_recall,adaptive_k_rag_200_k_50_model_4o_mini_with_buffer_v3-772915b6_precision,adaptive_k_rag_200_k_50_model_4o_mini_without_buffer_v1-e49a13e1_precision,seal_rag_200_k_5_model_4o_mini_v10-390ea694_precision,adaptive_k_rag_200_k_50_model_4o_mini_with_buffer_v3-772915b6_final_answer_correct,adaptive_k_rag_200_k_50_model_4o_mini_without_buffer_v1-e49a13e1_final_answer_correct,seal_rag_200_k_5_model_4o_mini_v10-390ea694_final_answer_correct
0,002de458-f6e2-4ef7-991f-6c31359ec71c,"{""context"": ""{'title': ['Richard Annesley, 3rd...","{""answer"": ""2 September 1770""}","{""docs_to_use"": [""Richard Annesley, 2nd Earl A...","{""docs_to_use"": [""Richard Annesley, 2nd Earl A...","{""docs_to_use"": [""George Annesley, 2nd Earl of...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,...,0.0000,1.0,0.5,0.00,0.2222,0.2500,0.0,1.0,0.0,0.0
1,009279eb-aa2f-4e9a-a388-0570b3d0fad7,"{""context"": ""{'title': ['Caspar Babypants', 'B...","{""answer"": ""Liverpool""}","{""docs_to_use"": [""God (John Lennon song): \""Go...","{""docs_to_use"": [""God (John Lennon song): \""Go...","{""docs_to_use"": [""God (John Lennon song): \""Go...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,...,0.6667,1.0,1.0,0.50,0.2857,1.0000,1.0,1.0,1.0,1.0
2,01afabb2-0984-4333-bfff-0b3ad45cc593,"{""context"": ""{'title': ['M. Venkataraju', 'Wor...","{""answer"": ""United States""}","{""docs_to_use"": [""World and Time Enough: World...","{""docs_to_use"": [""World and Time Enough: World...","{""docs_to_use"": [""World and Time Enough: World...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,...,0.6667,0.5,0.5,0.50,0.1667,1.0000,1.0,1.0,1.0,0.0
3,04491c40-8385-4c7c-9d07-665afd6ef52d,"{""context"": ""{'title': ['James Woolley (clockm...","{""answer"": ""Madame Pasca""}","{""docs_to_use"": [""James T. O'Donohoe: James T....","{""docs_to_use"": [""James T. O'Donohoe: James T....","{""docs_to_use"": [""James T. O'Donohoe: James T....","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,...,0.0000,1.0,1.0,0.00,0.2500,0.6667,0.0,1.0,1.0,1.0
4,097adb49-d636-4e90-b9cf-ffed265452f6,"{""context"": ""{'title': ['Takayama Tomoteru', '...","{""answer"": ""Galați""}","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ioan S. Nenițescu: Ioan S. N...","{""docs_to_use"": [""Ștefan I. Nenițescu: Ștefan ...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,...,0.6667,1.0,1.0,0.50,0.2857,1.0000,1.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,fc20c2aa-4e58-4517-ad1a-a050f0283457,"{""context"": ""{'title': ['Language Management',...","{""answ

In [37]:
df.columns.to_list()

['id',
 'inputs',
 'reference_outputs',
 'adaptive_k_rag_200_k_50_model_4o_mini_with_buffer_v3-772915b6_outputs',
 'adaptive_k_rag_200_k_50_model_4o_mini_without_buffer_v1-e49a13e1_outputs',
 'seal_rag_200_k_5_model_4o_mini_v10-390ea694_outputs',
 'adaptive_k_rag_200_k_50_model_4o_mini_with_buffer_v3-772915b6_run',
 'adaptive_k_rag_200_k_50_model_4o_mini_without_buffer_v1-e49a13e1_run',
 'seal_rag_200_k_5_model_4o_mini_v10-390ea694_run',
 'adaptive_k_rag_200_k_50_model_4o_mini_with_buffer_v3-772915b6_status',
 'adaptive_k_rag_200_k_50_model_4o_mini_without_buffer_v1-e49a13e1_status',
 'seal_rag_200_k_5_model_4o_mini_v10-390ea694_status',
 'adaptive_k_rag_200_k_50_model_4o_mini_with_buffer_v3-772915b6_error',
 'adaptive_k_rag_200_k_50_model_4o_mini_without_buffer_v1-e49a13e1_error',
 'seal_rag_200_k_5_model_4o_mini_v10-390ea694_error',
 'adaptive_k_rag_200_k_50_model_4o_mini_with_buffer_v3-772915b6_latency',
 'adaptive_k_rag_200_k_50_model_4o_mini_without_buffer_v1-e49a13e1_latency',
 '

In [39]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

# --- METRIC COMPARISON FUNCTIONS (REUSED) ---

def compare_binary_metrics(method1_results, method2_results, method1_name, method2_name):
    """Compare binary metrics using McNemar's test"""
    a1 = [int(x) for x in method1_results]
    a2 = [int(x) for x in method2_results]
    
    acc1 = sum(a1) / len(a1)
    acc2 = sum(a2) / len(a2)
    
    contingency_table = pd.crosstab(
        pd.Series(a1, name=method1_name), 
        pd.Series(a2, name=method2_name)
    )
    
    try:
        if contingency_table.shape == (2, 2):
            result = mcnemar(contingency_table, exact=False)
            p_value = result.pvalue
        else:
            p_value = 1.0
    except ValueError:
        p_value = 1.0
    
    return {
        'p_value': p_value,
        'is_significant': p_value < 0.05,
        'method1_accuracy': acc1,
        'method2_accuracy': acc2,
        'diff': acc1 - acc2
    }

def compare_continuous_metrics(method1_values, method2_values, method1_name, method2_name):
    """Compare continuous metrics using paired t-test"""
    v1 = np.array(method1_values, dtype=float)
    v2 = np.array(method2_values, dtype=float)
    
    mask = ~(np.isnan(v1) | np.isnan(v2))
    v1_clean = v1[mask]
    v2_clean = v2[mask]
    
    if len(v1_clean) == 0:
        return {'error': 'No valid data'}
    
    mean1 = np.mean(v1_clean)
    mean2 = np.mean(v2_clean)
    
    t_stat, p_value = stats.ttest_rel(v1_clean, v2_clean)
    
    return {
        'p_value': p_value,
        'is_significant': p_value < 0.05,
        'mean1': mean1,
        'mean2': mean2,
        'diff': mean1 - mean2
    }

# --- MAIN ANALYSIS FUNCTION FOR 4o-mini (UPDATED) ---

def analyze_seal_vs_adaptive_mini(csv_path):
    print(f"🚀 LOADING DATA from {csv_path}...")
    df = pd.read_csv(csv_path)
    
    # --- CONFIGURATION: Method Prefixes (UPDATED based on user input) ---
    methods = {
        "SEAL_RAG": "seal_rag_200_k_5_model_4o_mini_v10-390ea694",
        "Adaptive_K (Buffer)": "adaptive_k_rag_200_k_50_model_4o_mini_with_buffer_v3-772915b6",
        "Adaptive_K (No Buffer)": "adaptive_k_rag_200_k_50_model_4o_mini_without_buffer_v1-e49a13e1"
    }
    
    metrics = ["final_answer_correct", "precision", "recall", "f1"]
    
    print("\n📊 COMPARISON RESULTS (Base: SEAL_RAG [4o-mini])")
    print("=" * 100)
    
    base_name = "SEAL_RAG"
    base_prefix = methods[base_name]
    
    comparisons = [
        "Adaptive_K (Buffer)", 
        "Adaptive_K (No Buffer)"
    ]
    
    results_table = []
    
    for comp_name in comparisons:
        comp_prefix = methods[comp_name]
        print(f"\n⚔️  {base_name} vs {comp_name}")
        print("-" * 80)
        
        for metric in metrics:
            col_base = f"{base_prefix}_{metric}"
            col_comp = f"{comp_prefix}_{metric}"
            
            if col_base not in df.columns or col_comp not in df.columns:
                print(f"⚠️  Missing columns for {metric}")
                continue
                
            if metric == "final_answer_correct":
                res = compare_binary_metrics(df[col_base], df[col_comp], base_name, comp_name)
                significance = "**" if res['is_significant'] else ""
                print(f"   {metric:<20} | {base_name}: {res['method1_accuracy']:.1%} | {comp_name}: {res['method2_accuracy']:.1%} | Diff: {res['diff']:+.1%} | p={res['p_value']:.4f} {significance}")
                
                results_table.append({
                    "Comparison": f"{base_name} vs {comp_name}",
                    "Metric": "Accuracy",
                    "Base Score": res['method1_accuracy'],
                    "Comp Score": res['method2_accuracy'],
                    "Diff": res['diff'],
                    "P-Value": res['p_value'],
                    "Significant": res['is_significant']
                })
                
            else:
                res = compare_continuous_metrics(df[col_base], df[col_comp], base_name, comp_name)
                significance = "**" if res['is_significant'] else ""
                print(f"   {metric:<20} | {base_name}: {res['mean1']:.3f} | {comp_name}: {res['mean2']:.3f} | Diff: {res['diff']:+.3f} | p={res['p_value']:.4f} {significance}")

                results_table.append({
                    "Comparison": f"{base_name} vs {comp_name}",
                    "Metric": metric.capitalize(),
                    "Base Score": res['mean1'],
                    "Comp Score": res['mean2'],
                    "Diff": res['diff'],
                    "P-Value": res['p_value'],
                    "Significant": res['is_significant']
                })

    print("\n" + "=" * 100)
    return pd.DataFrame(results_table)

# --- USAGE ---
df_summary_mini = analyze_seal_vs_adaptive_mini("src/files/experiment_comparison/2WikiMultihopQA/adaptive_k_vs_seal_rag/seal_rag_4o_mini.csv")
print(df_summary_mini)

🚀 LOADING DATA from src/files/experiment_comparison/2WikiMultihopQA/adaptive_k_vs_seal_rag/seal_rag_4o_mini.csv...

📊 COMPARISON RESULTS (Base: SEAL_RAG [4o-mini])

⚔️  SEAL_RAG vs Adaptive_K (Buffer)
--------------------------------------------------------------------------------
   final_answer_correct | SEAL_RAG: 68.0% | Adaptive_K (Buffer): 60.5% | Diff: +7.5% | p=0.0778 
   precision            | SEAL_RAG: 0.887 | Adaptive_K (Buffer): 0.257 | Diff: +0.631 | p=0.0000 **
   recall               | SEAL_RAG: 0.454 | Adaptive_K (Buffer): 0.770 | Diff: -0.316 | p=0.0000 **
   f1                   | SEAL_RAG: 0.591 | Adaptive_K (Buffer): 0.377 | Diff: +0.214 | p=0.0000 **

⚔️  SEAL_RAG vs Adaptive_K (No Buffer)
--------------------------------------------------------------------------------
   final_answer_correct | SEAL_RAG: 68.0% | Adaptive_K (No Buffer): 40.5% | Diff: +27.5% | p=0.0000 **
   precision            | SEAL_RAG: 0.887 | Adaptive_K (No Buffer): 0.856 | Diff: +0.032 | p=0.13

## k = 1 | gpt 4.1-mini 

In [ ]:
df = pd.read_csv("files/experiment_comarison/k_1/seal_k_1_model_4_1_mini.csv")
#df = df.iloc[:, 30:]
df

,id,inputs,reference_outputs,crag_rag_k_1_model_gpt-4.1_mini_v_318-e38f155c_outputs,self_rag_k_1_model_gpt-4.1_mini_v_319-8f373d89_outputs,crag_rag_k_1_model_gpt-4.1_mini_v_318-e38f155c_run,self_rag_k_1_model_gpt-4.1_mini_v_319-8f373d89_run,crag_rag_k_1_model_gpt-4.1_mini_v_318-e38f155c_status,self_rag_k_1_model_gpt-4.1_mini_v_319-8f373d89_status,crag_rag_k_1_model_gpt-4.1_mini_v_318-e38f155c_error,...,basic_rag_k_1_model_gpt-4.1_mini_v_316-c794823e_total_cost,seal_rag_k_1_model_gpt-4.1_mini_v_316-9cbf4772_total_cost,basic_rag_k_1_model_gpt-4.1_mini_v_316-c794823e_final_answer_correct,seal_rag_k_1_model_gpt-4.1_mini_v_316-9cbf4772_final_answer_correct,basic_rag_k_1_model_gpt-4.1_mini_v_316-c794823e_f1,seal_rag_k_1_model_gpt-4.1_mini_v_316-9cbf4772_f1,basic_rag_k_1_model_gpt-4.1_mini_v_316-c794823e_recall,seal_rag_k_1_model_gpt-4.1_mini_v_316-9cbf4772_recall,basic_rag_k_1_model_gpt-4.1_mini_v_316-c794823e_precision,seal_rag_k_1_model_gpt-4.1_mini_v_316-9cbf4772_precision
0,0004211d-fdd0-46a4-ba01-3a5d916924e9,"{""context"": ""{'title': ['Clayton Oliver', 'Ste...","{""answer"": ""1.95 m""}","{""response"": ""195 cm"", ""retrieved_docs_list"": ...","{""response"": ""Unknown from context"", ""retrieve...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000094,0.004675,0,1,0.6667,0.6667,0.5,0.5,1,1.0
1,009df37c-3a8f-4a00-b8ed-342f7754257e,"{""context"": ""{'title': [\""2012\u201313 VCU Ram...","{""answer"": ""1838""}","{""response"": ""1968"", ""retrieved_docs_list"": [""...","{""response"": ""Not provided"", ""retrieved_docs_l...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000147,0.005162,0,1,0.6667,0.6667,0.5,0.5,1,1.0
2,00e0d7de-631c-47fc-b651-3e69c2029606,"{""context"": ""{'title': ['All I Wanna Do Is Mak...","{""answer"": ""2000""}","{""response"": ""March 14, 2000"", ""retrieved_docs...","{""response"": ""Not provided"", ""retrieved_docs_l...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000114,0.002539,1,1,0.6667,0.6667,0.5,0.5,1,1.0
3,0169d92a-754a-407c-afd8-5372b4b493a5,"{""context"": ""{'title': ['Alle tiders kupp', 'V...","{""answer"": ""Fomento Econ\u00f3mico Mexicano""}","{""response"": ""Coca-Cola FEMSA"", ""retrieved_doc...","{""response"": ""Coca-Cola FEMSA"", ""retrieved_doc...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000085,0.002220,0,0,0.6667,0.6667,0.5,0.5,1,1.0
4,016dfba5-d311-462a-8ce5-c9db85d2159e,"{""context"": ""{'title': ['Baidu Baike', 'Social...","{""answer"": ""Wikimedia Foundation""}","{""response"": ""Qihoo 360 operates the website r...","{""response"": ""Chinese Wikipedia"", ""retrieved_d...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000120,0.007985,0,0,0.6667,0.6667,0.5,0.5,1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,fdf4d662-d51a-4c36-a798-e5f9bc5bcc38,"{""context"": ""{'title': ['Archdeaconry of St An...","{""answer"": ""87""}","{""response"": ""87 parish churches"", ""retrieved_...","{""response"": ""Two parish churches"", ""retrieved...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000103,0.004451,0,0,0.6667,0.0000,0.5,0.0,1,0.0
996,fe72dc28-9b5f-41a7-ab23-bdb8e2ecaa97,"{""context"": ""{'title': ['Tralee Speedway', 'Di...","{""answer"": ""Dirt track racing""}","{""response"": ""Dirt track racing"", ""retrieved_d...","{""response"": ""Dirt track racing"", ""retrieved_d...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs

In [16]:
df.columns.to_list()

['id',
 'inputs',
 'reference_outputs',
 'crag_rag_k_1_model_gpt-4.1_mini_v_318-e38f155c_outputs',
 'self_rag_k_1_model_gpt-4.1_mini_v_319-8f373d89_outputs',
 'crag_rag_k_1_model_gpt-4.1_mini_v_318-e38f155c_run',
 'self_rag_k_1_model_gpt-4.1_mini_v_319-8f373d89_run',
 'crag_rag_k_1_model_gpt-4.1_mini_v_318-e38f155c_status',
 'self_rag_k_1_model_gpt-4.1_mini_v_319-8f373d89_status',
 'crag_rag_k_1_model_gpt-4.1_mini_v_318-e38f155c_error',
 'self_rag_k_1_model_gpt-4.1_mini_v_319-8f373d89_error',
 'crag_rag_k_1_model_gpt-4.1_mini_v_318-e38f155c_latency',
 'self_rag_k_1_model_gpt-4.1_mini_v_319-8f373d89_latency',
 'crag_rag_k_1_model_gpt-4.1_mini_v_318-e38f155c_tokens',
 'self_rag_k_1_model_gpt-4.1_mini_v_319-8f373d89_tokens',
 'crag_rag_k_1_model_gpt-4.1_mini_v_318-e38f155c_total_cost',
 'self_rag_k_1_model_gpt-4.1_mini_v_319-8f373d89_total_cost',
 'crag_rag_k_1_model_gpt-4.1_mini_v_318-e38f155c_precision',
 'self_rag_k_1_model_gpt-4.1_mini_v_319-8f373d89_precision',
 'crag_rag_k_1_model_g

In [17]:
# Run complete analysis
results = analyze_rag_experiments_complete(df)

# Access the summary table
summary_table = results['summary_table']
print(summary_table)

# Access detailed results
detailed_results = results['detailed_results']

# Save summary to CSV
#summary_table.to_csv('rag_comparison_summary.csv', index=False)

# Filter for significant results only
significant_only = summary_table[summary_table['Is_Significant'] == True]
print("\nSignificant results only:")
print(significant_only)

# Get specific comparison
seal_vs_crag_accuracy = summary_table[
    (summary_table['Comparison_Method'] == 'CRAG_RAG') & 
    (summary_table['Metric'] == 'Accuracy')
]

🚀 COMPREHENSIVE RAG METHOD COMPARISON
📊 BASE METHOD: SEAL RAG
🔍 COLUMN DETECTION RESULTS

📊 SEAL RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → seal_rag_k_1_model_gpt-4.1_mini_v_316-9cbf4772_final_answer_correct
   ✅ precision            → seal_rag_k_1_model_gpt-4.1_mini_v_316-9cbf4772_precision
   ✅ recall               → seal_rag_k_1_model_gpt-4.1_mini_v_316-9cbf4772_recall
   ✅ f1                   → seal_rag_k_1_model_gpt-4.1_mini_v_316-9cbf4772_f1

📊 SELF_RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → self_rag_k_1_model_gpt-4.1_mini_v_319-8f373d89_final_answer_correct
   ✅ precision            → self_rag_k_1_model_gpt-4.1_mini_v_319-8f373d89_precision
   ✅ recall               → self_rag_k_1_model_gpt-4.1_mini_v_319-8f373d89_recall
   ✅ f1                   → self_rag_k_1_model_gpt-4.1_mini_v_319-8f373d89_f1

📊 CRAG_RAG:
----------------------------

## k = 1 | gpt 4o-mini 

In [ ]:
df = pd.read_csv("files/experiment_comarison/k_1/seal_k_1_model_4_o_mini.csv")
#df = df.iloc[:, 30:]
df

,id,inputs,reference_outputs,basic_rag_k_1_model_gpt-4o_mini_v_315-57fe8196_outputs,self_rag_k_1_model_gpt-4o_mini_v_315-f5117a60_outputs,basic_rag_k_1_model_gpt-4o_mini_v_315-57fe8196_run,self_rag_k_1_model_gpt-4o_mini_v_315-f5117a60_run,basic_rag_k_1_model_gpt-4o_mini_v_315-57fe8196_status,self_rag_k_1_model_gpt-4o_mini_v_315-f5117a60_status,basic_rag_k_1_model_gpt-4o_mini_v_315-57fe8196_error,...,crag_rag_k_1_model_gpt-4o_mini_v_317-7c831dda_total_cost,seal_rag_k_1_model_gpt-4o_mini_v_315-6cc06d2d_total_cost,crag_rag_k_1_model_gpt-4o_mini_v_317-7c831dda_precision,seal_rag_k_1_model_gpt-4o_mini_v_315-6cc06d2d_precision,crag_rag_k_1_model_gpt-4o_mini_v_317-7c831dda_final_answer_correct,seal_rag_k_1_model_gpt-4o_mini_v_315-6cc06d2d_final_answer_correct,crag_rag_k_1_model_gpt-4o_mini_v_317-7c831dda_recall,seal_rag_k_1_model_gpt-4o_mini_v_315-6cc06d2d_recall,crag_rag_k_1_model_gpt-4o_mini_v_317-7c831dda_f1,seal_rag_k_1_model_gpt-4o_mini_v_315-6cc06d2d_f1
0,0004211d-fdd0-46a4-ba01-3a5d916924e9,"{""context"": ""{'title': ['Clayton Oliver', 'Ste...","{""answer"": ""1.95 m""}","{""response"": ""I don't know."", ""retrieved_docs_...","{""response"": ""Don't know."", ""retrieved_docs_li...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000265,0.001773,0,1.0,0,1,0.0,0.5,0.0000,0.6667
1,009df37c-3a8f-4a00-b8ed-342f7754257e,"{""context"": ""{'title': [\""2012\u201313 VCU Ram...","{""answer"": ""1838""}","{""response"": ""The year is not provided."", ""ret...","{""response"": ""Not specified in context."", ""ret...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000270,0.001872,0,1.0,1,1,0.0,0.5,0.0000,0.6667
2,00e0d7de-631c-47fc-b651-3e69c2029606,"{""context"": ""{'title': ['All I Wanna Do Is Mak...","{""answer"": ""2000""}","{""response"": ""March 14, 2000"", ""retrieved_docs...","{""response"": ""I don't know."", ""retrieved_docs_...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000346,0.003182,0,1.0,1,1,0.0,0.5,0.0000,0.6667
3,0169d92a-754a-407c-afd8-5372b4b493a5,"{""context"": ""{'title': ['Alle tiders kupp', 'V...","{""answer"": ""Fomento Econ\u00f3mico Mexicano""}","{""response"": ""Coca-Cola FEMSA"", ""retrieved_doc...","{""response"": ""Coca-Cola FEMSA"", ""retrieved_doc...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000066,0.000809,1,1.0,0,0,0.5,0.5,0.6667,0.6667
4,016dfba5-d311-462a-8ce5-c9db85d2159e,"{""context"": ""{'title': ['Baidu Baike', 'Social...","{""answer"": ""Wikimedia Foundation""}","{""response"": ""Baidu."", ""retrieved_docs_list"": ...","{""response"": ""Chinese government"", ""retrieved_...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000274,0.003052,0,1.0,0,0,0.0,0.5,0.0000,0.6667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,fdf4d662-d51a-4c36-a798-e5f9bc5bcc38,"{""context"": ""{'title': ['Archdeaconry of St An...","{""answer"": ""87""}","{""response"": ""The context does not specify the...","{""response"": ""Five parish churches."", ""retriev...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000285,0.001638,0,0.0,1,0,0.0,0.0,0.0000,0.0000
996,fe72dc28-9b5f-41a7-ab23-bdb8e2ecaa97,"{""context"": ""{'title': ['Tralee Speedway', 'Di...","{""answer"": ""Dirt track racing""}","{""response"": ""Dirt track racing"", ""retrieved_d...","{""response"": ""Dirt track racing"", ""retrieved_d...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""

In [19]:
df.columns.to_list()

['id',
 'inputs',
 'reference_outputs',
 'basic_rag_k_1_model_gpt-4o_mini_v_315-57fe8196_outputs',
 'self_rag_k_1_model_gpt-4o_mini_v_315-f5117a60_outputs',
 'basic_rag_k_1_model_gpt-4o_mini_v_315-57fe8196_run',
 'self_rag_k_1_model_gpt-4o_mini_v_315-f5117a60_run',
 'basic_rag_k_1_model_gpt-4o_mini_v_315-57fe8196_status',
 'self_rag_k_1_model_gpt-4o_mini_v_315-f5117a60_status',
 'basic_rag_k_1_model_gpt-4o_mini_v_315-57fe8196_error',
 'self_rag_k_1_model_gpt-4o_mini_v_315-f5117a60_error',
 'basic_rag_k_1_model_gpt-4o_mini_v_315-57fe8196_latency',
 'self_rag_k_1_model_gpt-4o_mini_v_315-f5117a60_latency',
 'basic_rag_k_1_model_gpt-4o_mini_v_315-57fe8196_tokens',
 'self_rag_k_1_model_gpt-4o_mini_v_315-f5117a60_tokens',
 'basic_rag_k_1_model_gpt-4o_mini_v_315-57fe8196_total_cost',
 'self_rag_k_1_model_gpt-4o_mini_v_315-f5117a60_total_cost',
 'basic_rag_k_1_model_gpt-4o_mini_v_315-57fe8196_recall',
 'self_rag_k_1_model_gpt-4o_mini_v_315-f5117a60_recall',
 'basic_rag_k_1_model_gpt-4o_mini_v_

In [20]:
# Run complete analysis
results = analyze_rag_experiments_complete(df)

# Access the summary table
summary_table = results['summary_table']
print(summary_table)

# Access detailed results
detailed_results = results['detailed_results']

# Save summary to CSV
#summary_table.to_csv('rag_comparison_summary.csv', index=False)

# Filter for significant results only
significant_only = summary_table[summary_table['Is_Significant'] == True]
print("\nSignificant results only:")
print(significant_only)

# Get specific comparison
seal_vs_crag_accuracy = summary_table[
    (summary_table['Comparison_Method'] == 'CRAG_RAG') & 
    (summary_table['Metric'] == 'Accuracy')
]

🚀 COMPREHENSIVE RAG METHOD COMPARISON
📊 BASE METHOD: SEAL RAG
🔍 COLUMN DETECTION RESULTS

📊 SEAL RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → seal_rag_k_1_model_gpt-4o_mini_v_315-6cc06d2d_final_answer_correct
   ✅ precision            → seal_rag_k_1_model_gpt-4o_mini_v_315-6cc06d2d_precision
   ✅ recall               → seal_rag_k_1_model_gpt-4o_mini_v_315-6cc06d2d_recall
   ✅ f1                   → seal_rag_k_1_model_gpt-4o_mini_v_315-6cc06d2d_f1

📊 SELF_RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → self_rag_k_1_model_gpt-4o_mini_v_315-f5117a60_final_answer_correct
   ✅ precision            → self_rag_k_1_model_gpt-4o_mini_v_315-f5117a60_precision
   ✅ recall               → self_rag_k_1_model_gpt-4o_mini_v_315-f5117a60_recall
   ✅ f1                   → self_rag_k_1_model_gpt-4o_mini_v_315-f5117a60_f1

📊 CRAG_RAG:
------------------------------------

## k = 1 | gpt 4o 

In [ ]:
df = pd.read_csv("files/experiment_comarison/k_1/seal_k_1_model_4_o.csv")
#df = df.iloc[:, 30:]
df

,id,inputs,reference_outputs,crag_rag_k_1_model_gpt-4o_v_315-7642cfdb_outputs,self_rag_k_1_model_gpt-4o_v_315-1b7e20ca_outputs,crag_rag_k_1_model_gpt-4o_v_315-7642cfdb_run,self_rag_k_1_model_gpt-4o_v_315-1b7e20ca_run,crag_rag_k_1_model_gpt-4o_v_315-7642cfdb_status,self_rag_k_1_model_gpt-4o_v_315-1b7e20ca_status,crag_rag_k_1_model_gpt-4o_v_315-7642cfdb_error,...,basic_rag_k_1_model_gpt-4o_v_315-bfa73703_total_cost,seal_rag_k_1_model_gpt-4o_v_314-9eca2761_total_cost,basic_rag_k_1_model_gpt-4o_v_315-bfa73703_f1,seal_rag_k_1_model_gpt-4o_v_314-9eca2761_f1,basic_rag_k_1_model_gpt-4o_v_315-bfa73703_precision,seal_rag_k_1_model_gpt-4o_v_314-9eca2761_precision,basic_rag_k_1_model_gpt-4o_v_315-bfa73703_final_answer_correct,seal_rag_k_1_model_gpt-4o_v_314-9eca2761_final_answer_correct,basic_rag_k_1_model_gpt-4o_v_315-bfa73703_recall,seal_rag_k_1_model_gpt-4o_v_314-9eca2761_recall
0,0004211d-fdd0-46a4-ba01-3a5d916924e9,"{""context"": ""{'title': ['Clayton Oliver', 'Ste...","{""answer"": ""1.95 m""}","{""response"": ""195 cm"", ""retrieved_docs_list"": ...","{""response"": ""I don't know."", ""retrieved_docs_...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000565,0.032835,0.6667,1.0000,1,1.0,0,1,0.5,1.0
1,009df37c-3a8f-4a00-b8ed-342f7754257e,"{""context"": ""{'title': [\""2012\u201313 VCU Ram...","{""answer"": ""1838""}","{""response"": ""1968"", ""retrieved_docs_list"": [""...","{""response"": ""1838"", ""retrieved_docs_list"": []...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000890,0.034978,0.6667,0.6667,1,1.0,0,0,0.5,0.5
2,00e0d7de-631c-47fc-b651-3e69c2029606,"{""context"": ""{'title': ['All I Wanna Do Is Mak...","{""answer"": ""2000""}","{""response"": ""January 2000"", ""retrieved_docs_l...","{""response"": ""I don't know."", ""retrieved_docs_...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000772,0.014955,0.6667,0.6667,1,1.0,1,1,0.5,0.5
3,0169d92a-754a-407c-afd8-5372b4b493a5,"{""context"": ""{'title': ['Alle tiders kupp', 'V...","{""answer"": ""Fomento Econ\u00f3mico Mexicano""}","{""response"": ""Coca-Cola FEMSA"", ""retrieved_doc...","{""response"": ""Coca-Cola FEMSA"", ""retrieved_doc...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000532,0.015510,0.6667,0.6667,1,1.0,0,0,0.5,0.5
4,016dfba5-d311-462a-8ce5-c9db85d2159e,"{""context"": ""{'title': ['Baidu Baike', 'Social...","{""answer"": ""Wikimedia Foundation""}","{""response"": ""ByteDance"", ""retrieved_docs_list...","{""response"": ""Wikipedia"", ""retrieved_docs_list...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000687,0.064340,0.6667,1.0000,1,1.0,0,1,0.5,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,fdf4d662-d51a-4c36-a798-e5f9bc5bcc38,"{""context"": ""{'title': ['Archdeaconry of St An...","{""answer"": ""87""}","{""response"": ""34 parish churches"", ""retrieved_...","{""response"": ""Eighty-nine churches"", ""retrieve...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000562,0.030558,0.6667,0.5000,1,0.5,0,0,0.5,0.5
996,fe72dc28-9b5f-41a7-ab23-bdb8e2ecaa97,"{""context"": ""{'title': ['Tralee Speedway', 'Di...","{""answer"": ""Dirt track racing""}","{""response"": ""Dirt track racing."", ""retrieved_...","{""response"": ""Dirt track racing"", ""retrieved_d...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000647,0.013848,0.6667,0.6667,1,1.0,1

In [23]:
df.columns.to_list()

['id',
 'inputs',
 'reference_outputs',
 'crag_rag_k_1_model_gpt-4o_v_315-7642cfdb_outputs',
 'self_rag_k_1_model_gpt-4o_v_315-1b7e20ca_outputs',
 'crag_rag_k_1_model_gpt-4o_v_315-7642cfdb_run',
 'self_rag_k_1_model_gpt-4o_v_315-1b7e20ca_run',
 'crag_rag_k_1_model_gpt-4o_v_315-7642cfdb_status',
 'self_rag_k_1_model_gpt-4o_v_315-1b7e20ca_status',
 'crag_rag_k_1_model_gpt-4o_v_315-7642cfdb_error',
 'self_rag_k_1_model_gpt-4o_v_315-1b7e20ca_error',
 'crag_rag_k_1_model_gpt-4o_v_315-7642cfdb_latency',
 'self_rag_k_1_model_gpt-4o_v_315-1b7e20ca_latency',
 'crag_rag_k_1_model_gpt-4o_v_315-7642cfdb_tokens',
 'self_rag_k_1_model_gpt-4o_v_315-1b7e20ca_tokens',
 'crag_rag_k_1_model_gpt-4o_v_315-7642cfdb_total_cost',
 'self_rag_k_1_model_gpt-4o_v_315-1b7e20ca_total_cost',
 'crag_rag_k_1_model_gpt-4o_v_315-7642cfdb_final_answer_correct',
 'self_rag_k_1_model_gpt-4o_v_315-1b7e20ca_final_answer_correct',
 'crag_rag_k_1_model_gpt-4o_v_315-7642cfdb_f1',
 'self_rag_k_1_model_gpt-4o_v_315-1b7e20ca_f1',


In [24]:
# Run complete analysis
results = analyze_rag_experiments_complete(df)

# Access the summary table
summary_table = results['summary_table']
print(summary_table)

# Access detailed results
detailed_results = results['detailed_results']

# Save summary to CSV
#summary_table.to_csv('rag_comparison_summary.csv', index=False)

# Filter for significant results only
significant_only = summary_table[summary_table['Is_Significant'] == True]
print("\nSignificant results only:")
print(significant_only)

# Get specific comparison
seal_vs_crag_accuracy = summary_table[
    (summary_table['Comparison_Method'] == 'CRAG_RAG') & 
    (summary_table['Metric'] == 'Accuracy')
]

🚀 COMPREHENSIVE RAG METHOD COMPARISON
📊 BASE METHOD: SEAL RAG
🔍 COLUMN DETECTION RESULTS

📊 SEAL RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → seal_rag_k_1_model_gpt-4o_v_314-9eca2761_final_answer_correct
   ✅ precision            → seal_rag_k_1_model_gpt-4o_v_314-9eca2761_precision
   ✅ recall               → seal_rag_k_1_model_gpt-4o_v_314-9eca2761_recall
   ✅ f1                   → seal_rag_k_1_model_gpt-4o_v_314-9eca2761_f1

📊 SELF_RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → self_rag_k_1_model_gpt-4o_v_315-1b7e20ca_final_answer_correct
   ✅ precision            → self_rag_k_1_model_gpt-4o_v_315-1b7e20ca_precision
   ✅ recall               → self_rag_k_1_model_gpt-4o_v_315-1b7e20ca_recall
   ✅ f1                   → self_rag_k_1_model_gpt-4o_v_315-1b7e20ca_f1

📊 CRAG_RAG:
----------------------------------------------------------------------------

## k = 1 | gpt 4.1

In [ ]:
df = pd.read_csv("files/experiment_comarison/k_1/seal_k_1_model_4_1.csv")
#df = df.iloc[:, 30:]
df

,id,inputs,reference_outputs,seal_rag_k_1_model_gpt-4.1_v_313-05674c6f_outputs,self_rag_k_1_model_gpt-4.1_v_313-68bb3abd_outputs,seal_rag_k_1_model_gpt-4.1_v_313-05674c6f_run,self_rag_k_1_model_gpt-4.1_v_313-68bb3abd_run,seal_rag_k_1_model_gpt-4.1_v_313-05674c6f_status,self_rag_k_1_model_gpt-4.1_v_313-68bb3abd_status,seal_rag_k_1_model_gpt-4.1_v_313-05674c6f_error,...,basic_rag_k_1_model_gpt-4.1_v_313-19ef35d4_total_cost,crag_rag_k_1_model_gpt-4.1_v_313-5fe8c082_total_cost,basic_rag_k_1_model_gpt-4.1_v_313-19ef35d4_final_answer_correct,crag_rag_k_1_model_gpt-4.1_v_313-5fe8c082_final_answer_correct,basic_rag_k_1_model_gpt-4.1_v_313-19ef35d4_precision,crag_rag_k_1_model_gpt-4.1_v_313-5fe8c082_precision,basic_rag_k_1_model_gpt-4.1_v_313-19ef35d4_f1,crag_rag_k_1_model_gpt-4.1_v_313-5fe8c082_f1,basic_rag_k_1_model_gpt-4.1_v_313-19ef35d4_recall,crag_rag_k_1_model_gpt-4.1_v_313-5fe8c082_recall
0,0004211d-fdd0-46a4-ba01-3a5d916924e9,"{""context"": ""{'title': ['Clayton Oliver', 'Ste...","{""answer"": ""1.95 m""}","{""response"": ""1.95 m"", ""retrieved_docs_list"": ...","{""response"": ""1.95 m"", ""retrieved_docs_list"": ...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000484,0.003738,0,1,1,0,0.6667,0.0000,0.5,0.0
1,009df37c-3a8f-4a00-b8ed-342f7754257e,"{""context"": ""{'title': [\""2012\u201313 VCU Ram...","{""answer"": ""1838""}","{""response"": ""1968"", ""retrieved_docs_list"": [""...","{""response"": ""1968"", ""retrieved_docs_list"": [""...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000752,0.004128,0,0,1,0,0.6667,0.0000,0.5,0.0
2,00e0d7de-631c-47fc-b651-3e69c2029606,"{""context"": ""{'title': ['All I Wanna Do Is Mak...","{""answer"": ""2000""}","{""response"": ""March 14, 2000"", ""retrieved_docs...","{""response"": ""March 14, 2000"", ""retrieved_docs...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000610,0.004146,1,1,1,0,0.6667,0.0000,0.5,0.0
3,0169d92a-754a-407c-afd8-5372b4b493a5,"{""context"": ""{'title': ['Alle tiders kupp', 'V...","{""answer"": ""Fomento Econ\u00f3mico Mexicano""}","{""response"": ""Coca-Cola FEMSA"", ""retrieved_doc...","{""response"": ""Coca-Cola FEMSA"", ""retrieved_doc...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000426,0.000876,0,0,1,1,0.6667,0.6667,0.5,0.5
4,016dfba5-d311-462a-8ce5-c9db85d2159e,"{""context"": ""{'title': ['Baidu Baike', 'Social...","{""answer"": ""Wikimedia Foundation""}","{""response"": ""Wikimedia Foundation"", ""retrieve...","{""response"": ""Chinese Wikipedia"", ""retrieved_d...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000542,0.003744,0,0,1,0,0.6667,0.0000,0.5,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,fdf4d662-d51a-4c36-a798-e5f9bc5bcc38,"{""context"": ""{'title': ['Archdeaconry of St An...","{""answer"": ""87""}","{""response"": ""87 parish churches"", ""retrieved_...","{""response"": ""Eighty-seven churches"", ""retriev...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000466,0.003552,0,1,1,0,0.6667,0.0000,0.5,0.0
996,fe72dc28-9b5f-41a7-ab23-bdb8e2ecaa97,"{""context"": ""{'title': ['Tralee Speedway', 'Di...","{""answer"": ""Dirt track racing""}","{""response"": ""Dirt track racing"", ""retrieved_d...","{""response"": ""Dirt track racing"", ""retrieved_d...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000510,0.001052,1,1,1,1,0.6667

In [26]:
df.columns.to_list()

['id',
 'inputs',
 'reference_outputs',
 'seal_rag_k_1_model_gpt-4.1_v_313-05674c6f_outputs',
 'self_rag_k_1_model_gpt-4.1_v_313-68bb3abd_outputs',
 'seal_rag_k_1_model_gpt-4.1_v_313-05674c6f_run',
 'self_rag_k_1_model_gpt-4.1_v_313-68bb3abd_run',
 'seal_rag_k_1_model_gpt-4.1_v_313-05674c6f_status',
 'self_rag_k_1_model_gpt-4.1_v_313-68bb3abd_status',
 'seal_rag_k_1_model_gpt-4.1_v_313-05674c6f_error',
 'self_rag_k_1_model_gpt-4.1_v_313-68bb3abd_error',
 'seal_rag_k_1_model_gpt-4.1_v_313-05674c6f_latency',
 'self_rag_k_1_model_gpt-4.1_v_313-68bb3abd_latency',
 'seal_rag_k_1_model_gpt-4.1_v_313-05674c6f_tokens',
 'self_rag_k_1_model_gpt-4.1_v_313-68bb3abd_tokens',
 'seal_rag_k_1_model_gpt-4.1_v_313-05674c6f_total_cost',
 'self_rag_k_1_model_gpt-4.1_v_313-68bb3abd_total_cost',
 'seal_rag_k_1_model_gpt-4.1_v_313-05674c6f_recall',
 'self_rag_k_1_model_gpt-4.1_v_313-68bb3abd_recall',
 'seal_rag_k_1_model_gpt-4.1_v_313-05674c6f_precision',
 'self_rag_k_1_model_gpt-4.1_v_313-68bb3abd_precisio

In [27]:
# Run complete analysis
results = analyze_rag_experiments_complete(df)

# Access the summary table
summary_table = results['summary_table']
print(summary_table)

# Access detailed results
detailed_results = results['detailed_results']

# Save summary to CSV
#summary_table.to_csv('rag_comparison_summary.csv', index=False)

# Filter for significant results only
significant_only = summary_table[summary_table['Is_Significant'] == True]
print("\nSignificant results only:")
print(significant_only)

# Get specific comparison
seal_vs_crag_accuracy = summary_table[
    (summary_table['Comparison_Method'] == 'CRAG_RAG') & 
    (summary_table['Metric'] == 'Accuracy')
]

🚀 COMPREHENSIVE RAG METHOD COMPARISON
📊 BASE METHOD: SEAL RAG
🔍 COLUMN DETECTION RESULTS

📊 SEAL RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → seal_rag_k_1_model_gpt-4.1_v_313-05674c6f_final_answer_correct
   ✅ precision            → seal_rag_k_1_model_gpt-4.1_v_313-05674c6f_precision
   ✅ recall               → seal_rag_k_1_model_gpt-4.1_v_313-05674c6f_recall
   ✅ f1                   → seal_rag_k_1_model_gpt-4.1_v_313-05674c6f_f1

📊 SELF_RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → self_rag_k_1_model_gpt-4.1_v_313-68bb3abd_final_answer_correct
   ✅ precision            → self_rag_k_1_model_gpt-4.1_v_313-68bb3abd_precision
   ✅ recall               → self_rag_k_1_model_gpt-4.1_v_313-68bb3abd_recall
   ✅ f1                   → self_rag_k_1_model_gpt-4.1_v_313-68bb3abd_f1

📊 CRAG_RAG:
--------------------------------------------------------------------

## k = 3 | gpt 4.1-mini 

In [ ]:
df = pd.read_csv("files/experiment_comarison/k_3/seal_k_3_model_4_1_mini.csv")
#df = df.iloc[:, 30:]
df

,id,inputs,reference_outputs,crag_rag_k_3_model_gpt-4.1_mini_v_311-85775432_outputs,seal_rag_k_3_model_gpt-4.1_mini_v_311-bca1ff82_outputs,crag_rag_k_3_model_gpt-4.1_mini_v_311-85775432_run,seal_rag_k_3_model_gpt-4.1_mini_v_311-bca1ff82_run,crag_rag_k_3_model_gpt-4.1_mini_v_311-85775432_status,seal_rag_k_3_model_gpt-4.1_mini_v_311-bca1ff82_status,crag_rag_k_3_model_gpt-4.1_mini_v_311-85775432_error,...,basic_rag_k_3_model_gpt-4.1_mini_v_309-1005c6f1_total_cost,self_rag_k_3_model_gpt-4.1_mini_v_310-0ebf7a92_total_cost,basic_rag_k_3_model_gpt-4.1_mini_v_309-1005c6f1_recall,self_rag_k_3_model_gpt-4.1_mini_v_310-0ebf7a92_recall,basic_rag_k_3_model_gpt-4.1_mini_v_309-1005c6f1_f1,self_rag_k_3_model_gpt-4.1_mini_v_310-0ebf7a92_f1,basic_rag_k_3_model_gpt-4.1_mini_v_309-1005c6f1_precision,self_rag_k_3_model_gpt-4.1_mini_v_310-0ebf7a92_precision,basic_rag_k_3_model_gpt-4.1_mini_v_309-1005c6f1_final_answer_correct,self_rag_k_3_model_gpt-4.1_mini_v_310-0ebf7a92_final_answer_correct
0,0004211d-fdd0-46a4-ba01-3a5d916924e9,"{""context"": ""{'title': ['Clayton Oliver', 'Ste...","{""answer"": ""1.95 m""}","{""response"": ""195 cm"", ""retrieved_docs_list"": ...","{""response"": ""1.95 m"", ""retrieved_docs_list"": ...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000193,0.001798,0.5,0.0,0.4,0.0000,0.3333,0.0000,0,0
1,009df37c-3a8f-4a00-b8ed-342f7754257e,"{""context"": ""{'title': [\""2012\u201313 VCU Ram...","{""answer"": ""1838""}","{""response"": ""1968"", ""retrieved_docs_list"": [""...","{""response"": ""1968"", ""retrieved_docs_list"": [""...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000296,0.001472,0.5,0.0,0.4,0.0000,0.3333,0.0000,0,0
2,00e0d7de-631c-47fc-b651-3e69c2029606,"{""context"": ""{'title': ['All I Wanna Do Is Mak...","{""answer"": ""2000""}","{""response"": ""March 14, 2000"", ""retrieved_docs...","{""response"": ""March 14, 2000"", ""retrieved_docs...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000237,0.002019,1.0,0.0,0.8,0.0000,0.6667,0.0000,1,0
3,0169d92a-754a-407c-afd8-5372b4b493a5,"{""context"": ""{'title': ['Alle tiders kupp', 'V...","{""answer"": ""Fomento Econ\u00f3mico Mexicano""}","{""response"": ""Coca-Cola FEMSA"", ""retrieved_doc...","{""response"": ""Coca-Cola FEMSA"", ""retrieved_doc...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000171,0.000828,1.0,1.0,0.8,0.8000,0.6667,0.6667,0,0
4,016dfba5-d311-462a-8ce5-c9db85d2159e,"{""context"": ""{'title': ['Baidu Baike', 'Social...","{""answer"": ""Wikimedia Foundation""}","{""response"": ""Wikipedia is operated by an indi...","{""response"": ""Pan Haidong"", ""retrieved_docs_li...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000164,0.001251,0.5,0.5,0.4,0.6667,0.3333,1.0000,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,fdf4d662-d51a-4c36-a798-e5f9bc5bcc38,"{""context"": ""{'title': ['Archdeaconry of St An...","{""answer"": ""87""}","{""response"": ""87 parish churches"", ""retrieved_...","{""response"": ""Eighty-eight parish churches"", ""...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000145,0.000617,0.5,0.0,0.4,0.0000,0.3333,0.0000,0,0
996,fe72dc28-9b5f-41a7-ab23-bdb8e2ecaa97,"{""context"": ""{'title': ['Tralee Speedway', 'Di...","{""answer"": ""Dirt track racing""}","{""response"": ""Dirt track racing on clay or dir...","{""response"": ""Dirt track racing"", ""retrieved_d...","{""inputs"": {""inputs"": {""context"": ""{'title'

In [30]:
df.columns.to_list()

['id',
 'inputs',
 'reference_outputs',
 'crag_rag_k_3_model_gpt-4.1_mini_v_311-85775432_outputs',
 'seal_rag_k_3_model_gpt-4.1_mini_v_311-bca1ff82_outputs',
 'crag_rag_k_3_model_gpt-4.1_mini_v_311-85775432_run',
 'seal_rag_k_3_model_gpt-4.1_mini_v_311-bca1ff82_run',
 'crag_rag_k_3_model_gpt-4.1_mini_v_311-85775432_status',
 'seal_rag_k_3_model_gpt-4.1_mini_v_311-bca1ff82_status',
 'crag_rag_k_3_model_gpt-4.1_mini_v_311-85775432_error',
 'seal_rag_k_3_model_gpt-4.1_mini_v_311-bca1ff82_error',
 'crag_rag_k_3_model_gpt-4.1_mini_v_311-85775432_latency',
 'seal_rag_k_3_model_gpt-4.1_mini_v_311-bca1ff82_latency',
 'crag_rag_k_3_model_gpt-4.1_mini_v_311-85775432_tokens',
 'seal_rag_k_3_model_gpt-4.1_mini_v_311-bca1ff82_tokens',
 'crag_rag_k_3_model_gpt-4.1_mini_v_311-85775432_total_cost',
 'seal_rag_k_3_model_gpt-4.1_mini_v_311-bca1ff82_total_cost',
 'crag_rag_k_3_model_gpt-4.1_mini_v_311-85775432_final_answer_correct',
 'seal_rag_k_3_model_gpt-4.1_mini_v_311-bca1ff82_final_answer_correct',


In [31]:
# Run complete analysis
results = analyze_rag_experiments_complete(df)

# Access the summary table
summary_table = results['summary_table']
print(summary_table)

# Access detailed results
detailed_results = results['detailed_results']

# Save summary to CSV
#summary_table.to_csv('rag_comparison_summary.csv', index=False)

# Filter for significant results only
significant_only = summary_table[summary_table['Is_Significant'] == True]
print("\nSignificant results only:")
print(significant_only)

# Get specific comparison
seal_vs_crag_accuracy = summary_table[
    (summary_table['Comparison_Method'] == 'CRAG_RAG') & 
    (summary_table['Metric'] == 'Accuracy')
]

🚀 COMPREHENSIVE RAG METHOD COMPARISON
📊 BASE METHOD: SEAL RAG
🔍 COLUMN DETECTION RESULTS

📊 SEAL RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → seal_rag_k_3_model_gpt-4.1_mini_v_311-bca1ff82_final_answer_correct
   ✅ precision            → seal_rag_k_3_model_gpt-4.1_mini_v_311-bca1ff82_precision
   ✅ recall               → seal_rag_k_3_model_gpt-4.1_mini_v_311-bca1ff82_recall
   ✅ f1                   → seal_rag_k_3_model_gpt-4.1_mini_v_311-bca1ff82_f1

📊 SELF_RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → self_rag_k_3_model_gpt-4.1_mini_v_310-0ebf7a92_final_answer_correct
   ✅ precision            → self_rag_k_3_model_gpt-4.1_mini_v_310-0ebf7a92_precision
   ✅ recall               → self_rag_k_3_model_gpt-4.1_mini_v_310-0ebf7a92_recall
   ✅ f1                   → self_rag_k_3_model_gpt-4.1_mini_v_310-0ebf7a92_f1

📊 CRAG_RAG:
----------------------------

## k = 3 | gpt 4o-mini 

In [ ]:
df = pd.read_csv("files/experiment_comarison/k_3/seal_k_3_model_4_o_mini.csv")
#df = df.iloc[:, 30:]
df

,id,inputs,reference_outputs,crag_rag_k_3_model_gpt-4o_mini_v_313-bf757073_outputs,self_rag_k_3_model_gpt-4o_mini_v_312-d812c7b7_outputs,crag_rag_k_3_model_gpt-4o_mini_v_313-bf757073_run,self_rag_k_3_model_gpt-4o_mini_v_312-d812c7b7_run,crag_rag_k_3_model_gpt-4o_mini_v_313-bf757073_status,self_rag_k_3_model_gpt-4o_mini_v_312-d812c7b7_status,crag_rag_k_3_model_gpt-4o_mini_v_313-bf757073_error,...,basic_rag_k_3_model_gpt-4o_mini_v_312-f0d9fd12_total_cost,seal_rag_k_3_model_gpt-4o_mini_v_311-56340dc9_total_cost,basic_rag_k_3_model_gpt-4o_mini_v_312-f0d9fd12_precision,seal_rag_k_3_model_gpt-4o_mini_v_311-56340dc9_precision,basic_rag_k_3_model_gpt-4o_mini_v_312-f0d9fd12_recall,seal_rag_k_3_model_gpt-4o_mini_v_311-56340dc9_recall,basic_rag_k_3_model_gpt-4o_mini_v_312-f0d9fd12_final_answer_correct,seal_rag_k_3_model_gpt-4o_mini_v_311-56340dc9_final_answer_correct,basic_rag_k_3_model_gpt-4o_mini_v_312-f0d9fd12_f1,seal_rag_k_3_model_gpt-4o_mini_v_311-56340dc9_f1
0,0004211d-fdd0-46a4-ba01-3a5d916924e9,"{""context"": ""{'title': ['Clayton Oliver', 'Ste...","{""answer"": ""1.95 m""}","{""response"": ""I don't know."", ""retrieved_docs_...","{""response"": ""Don't know."", ""retrieved_docs_li...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000072,0.002544,0.3333,1.0,0.5,0.5,0,1,0.4,0.6667
1,009df37c-3a8f-4a00-b8ed-342f7754257e,"{""context"": ""{'title': [\""2012\u201313 VCU Ram...","{""answer"": ""1838""}","{""response"": ""1968"", ""retrieved_docs_list"": [""...","{""response"": ""Not specified in context."", ""ret...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000111,0.002888,0.3333,1.0,0.5,0.5,0,1,0.4,0.6667
2,00e0d7de-631c-47fc-b651-3e69c2029606,"{""context"": ""{'title': ['All I Wanna Do Is Mak...","{""answer"": ""2000""}","{""response"": ""March 14, 2000"", ""retrieved_docs...","{""response"": ""Don't know."", ""retrieved_docs_li...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000092,0.004289,0.6667,1.0,1.0,0.5,1,0,0.8,0.6667
3,0169d92a-754a-407c-afd8-5372b4b493a5,"{""context"": ""{'title': ['Alle tiders kupp', 'V...","{""answer"": ""Fomento Econ\u00f3mico Mexicano""}","{""response"": ""Coca-Cola FEMSA"", ""retrieved_doc...","{""response"": ""Coca-Cola FEMSA"", ""retrieved_doc...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000064,0.001363,0.6667,1.0,1.0,0.5,0,0,0.8,0.6667
4,016dfba5-d311-462a-8ce5-c9db85d2159e,"{""context"": ""{'title': ['Baidu Baike', 'Social...","{""answer"": ""Wikimedia Foundation""}","{""response"": ""Baike.com, operated by Pan Haido...","{""response"": ""Chinese Wikipedia"", ""retrieved_d...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000062,0.003356,0.3333,1.0,0.5,0.5,0,0,0.4,0.6667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,fdf4d662-d51a-4c36-a798-e5f9bc5bcc38,"{""context"": ""{'title': ['Archdeaconry of St An...","{""answer"": ""87""}","{""response"": ""Eighty-eight parish churches."", ...","{""response"": ""Eighty-eight parish churches."", ...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.000054,0.001163,0.3333,0.0,0.5,0.0,0,0,0.4,0.0000
996,fe72dc28-9b5f-41a7-ab23-bdb8e2ecaa97,"{""context"": ""{'title': ['Tralee Speedway', 'Di...","{""answer"": ""Dirt track racing""}","{""response"": ""Dirt track racing"", ""retrieved_d...","{""response"": ""Dirt track racing"", ""retrieved_d...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""c

In [33]:
df.columns.to_list()

['id',
 'inputs',
 'reference_outputs',
 'crag_rag_k_3_model_gpt-4o_mini_v_313-bf757073_outputs',
 'self_rag_k_3_model_gpt-4o_mini_v_312-d812c7b7_outputs',
 'crag_rag_k_3_model_gpt-4o_mini_v_313-bf757073_run',
 'self_rag_k_3_model_gpt-4o_mini_v_312-d812c7b7_run',
 'crag_rag_k_3_model_gpt-4o_mini_v_313-bf757073_status',
 'self_rag_k_3_model_gpt-4o_mini_v_312-d812c7b7_status',
 'crag_rag_k_3_model_gpt-4o_mini_v_313-bf757073_error',
 'self_rag_k_3_model_gpt-4o_mini_v_312-d812c7b7_error',
 'crag_rag_k_3_model_gpt-4o_mini_v_313-bf757073_latency',
 'self_rag_k_3_model_gpt-4o_mini_v_312-d812c7b7_latency',
 'crag_rag_k_3_model_gpt-4o_mini_v_313-bf757073_tokens',
 'self_rag_k_3_model_gpt-4o_mini_v_312-d812c7b7_tokens',
 'crag_rag_k_3_model_gpt-4o_mini_v_313-bf757073_total_cost',
 'self_rag_k_3_model_gpt-4o_mini_v_312-d812c7b7_total_cost',
 'crag_rag_k_3_model_gpt-4o_mini_v_313-bf757073_final_answer_correct',
 'self_rag_k_3_model_gpt-4o_mini_v_312-d812c7b7_final_answer_correct',
 'crag_rag_k_3_m

In [34]:
# Run complete analysis
results = analyze_rag_experiments_complete(df)

# Access the summary table
summary_table = results['summary_table']
print(summary_table)

# Access detailed results
detailed_results = results['detailed_results']

# Save summary to CSV
#summary_table.to_csv('rag_comparison_summary.csv', index=False)

# Filter for significant results only
significant_only = summary_table[summary_table['Is_Significant'] == True]
print("\nSignificant results only:")
print(significant_only)

# Get specific comparison
seal_vs_crag_accuracy = summary_table[
    (summary_table['Comparison_Method'] == 'CRAG_RAG') & 
    (summary_table['Metric'] == 'Accuracy')
]

🚀 COMPREHENSIVE RAG METHOD COMPARISON
📊 BASE METHOD: SEAL RAG
🔍 COLUMN DETECTION RESULTS

📊 SEAL RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → seal_rag_k_3_model_gpt-4o_mini_v_311-56340dc9_final_answer_correct
   ✅ precision            → seal_rag_k_3_model_gpt-4o_mini_v_311-56340dc9_precision
   ✅ recall               → seal_rag_k_3_model_gpt-4o_mini_v_311-56340dc9_recall
   ✅ f1                   → seal_rag_k_3_model_gpt-4o_mini_v_311-56340dc9_f1

📊 SELF_RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → self_rag_k_3_model_gpt-4o_mini_v_312-d812c7b7_final_answer_correct
   ✅ precision            → self_rag_k_3_model_gpt-4o_mini_v_312-d812c7b7_precision
   ✅ recall               → self_rag_k_3_model_gpt-4o_mini_v_312-d812c7b7_recall
   ✅ f1                   → self_rag_k_3_model_gpt-4o_mini_v_312-d812c7b7_f1

📊 CRAG_RAG:
------------------------------------

## k = 3 | gpt 4o

In [ ]:
df = pd.read_csv("files/experiment_comarison/k_3/seal_k_3_model_4_o.csv")
#df = df.iloc[:, 30:]
df

,id,inputs,reference_outputs,seal_rag_k_3_model_gpt-4o_max_loop_5_v200_old_1-24a0719b_outputs,self_rag_k_3_model_gpt-4o_v_304-1cb5682e_outputs,seal_rag_k_3_model_gpt-4o_max_loop_5_v200_old_1-24a0719b_run,self_rag_k_3_model_gpt-4o_v_304-1cb5682e_run,seal_rag_k_3_model_gpt-4o_max_loop_5_v200_old_1-24a0719b_status,self_rag_k_3_model_gpt-4o_v_304-1cb5682e_status,seal_rag_k_3_model_gpt-4o_max_loop_5_v200_old_1-24a0719b_error,...,basic_rag_k_3_model_gpt-4o_max_loop_5_v2-34ab6a2c_wrapper,crag_rag_k_3_model_gpt-4o_max_loop_5_v2-03a122c5_wrapper,basic_rag_k_3_model_gpt-4o_max_loop_5_v2-34ab6a2c_final_answer_correct,crag_rag_k_3_model_gpt-4o_max_loop_5_v2-03a122c5_final_answer_correct,basic_rag_k_3_model_gpt-4o_max_loop_5_v2-34ab6a2c_precision,crag_rag_k_3_model_gpt-4o_max_loop_5_v2-03a122c5_precision,basic_rag_k_3_model_gpt-4o_max_loop_5_v2-34ab6a2c_recall,crag_rag_k_3_model_gpt-4o_max_loop_5_v2-03a122c5_recall,basic_rag_k_3_model_gpt-4o_max_loop_5_v2-34ab6a2c_f1,crag_rag_k_3_model_gpt-4o_max_loop_5_v2-03a122c5_f1
0,0004211d-fdd0-46a4-ba01-3a5d916924e9,"{""context"": ""{'title': ['Clayton Oliver', 'Ste...","{""answer"": ""1.95 m""}","{""response"": ""1.95 m"", ""retrieved_docs_list"": ...","{""response"": ""I don't know."", ""retrieved_docs_...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,NaN,NaN,0,1.0,0.3333,0.0,0.5,0.0,0.4,0.0
1,009df37c-3a8f-4a00-b8ed-342f7754257e,"{""context"": ""{'title': [\""2012\u201313 VCU Ram...","{""answer"": ""1838""}","{""response"": ""1838"", ""retrieved_docs_list"": [""...","{""response"": ""1968"", ""retrieved_docs_list"": []...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,NaN,NaN,0,0.0,0.3333,0.0,0.5,0.0,0.4,0.0
2,00e0d7de-631c-47fc-b651-3e69c2029606,"{""context"": ""{'title': ['All I Wanna Do Is Mak...","{""answer"": ""2000""}","{""response"": ""March 14, 2000"", ""retrieved_docs...","{""response"": ""March 14, 2000"", ""retrieved_docs...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,NaN,NaN,1,1.0,0.6667,0.0,1.0,0.0,0.8,0.0
3,0169d92a-754a-407c-afd8-5372b4b493a5,"{""context"": ""{'title': ['Alle tiders kupp', 'V...","{""answer"": ""Fomento Econ\u00f3mico Mexicano""}","{""response"": ""Coca-Cola FEMSA"", ""retrieved_doc...","{""response"": ""Coca-Cola FEMSA"", ""retrieved_doc...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,NaN,NaN,0,0.0,0.6667,0.5,1.0,0.5,0.8,0.5
4,016dfba5-d311-462a-8ce5-c9db85d2159e,"{""context"": ""{'title': ['Baidu Baike', 'Social...","{""answer"": ""Wikimedia Foundation""}","{""response"": ""Pan Haidong"", ""retrieved_docs_li...","{""response"": ""Chinese Wikipedia"", ""retrieved_d...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,NaN,NaN,0,0.0,0.3333,0.0,0.5,0.0,0.4,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,fdf4d662-d51a-4c36-a798-e5f9bc5bcc38,"{""context"": ""{'title': ['Archdeaconry of St An...","{""answer"": ""87""}","{""response"": ""Eighty-eight churches."", ""retrie...","{""response"": ""Eighty-eight parish churches."", ...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,NaN,NaN,0,1.0,0.3333,0.0,0.5,0.0,0.4,0.0
996,fe72dc28-9b5f-41a7-ab23-bdb8e2ecaa97,"{""context"": ""{'title': ['Tralee Speedway', 'Di...","{""answer"": ""Dirt track racing""}","{""response"": ""Dirt track racing."", ""retrieved_...","{""response"": ""Dirt track racing"", ""retrieved_d...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context""

In [36]:
df.columns.to_list()

['id',
 'inputs',
 'reference_outputs',
 'seal_rag_k_3_model_gpt-4o_max_loop_5_v200_old_1-24a0719b_outputs',
 'self_rag_k_3_model_gpt-4o_v_304-1cb5682e_outputs',
 'seal_rag_k_3_model_gpt-4o_max_loop_5_v200_old_1-24a0719b_run',
 'self_rag_k_3_model_gpt-4o_v_304-1cb5682e_run',
 'seal_rag_k_3_model_gpt-4o_max_loop_5_v200_old_1-24a0719b_status',
 'self_rag_k_3_model_gpt-4o_v_304-1cb5682e_status',
 'seal_rag_k_3_model_gpt-4o_max_loop_5_v200_old_1-24a0719b_error',
 'self_rag_k_3_model_gpt-4o_v_304-1cb5682e_error',
 'seal_rag_k_3_model_gpt-4o_max_loop_5_v200_old_1-24a0719b_latency',
 'self_rag_k_3_model_gpt-4o_v_304-1cb5682e_latency',
 'seal_rag_k_3_model_gpt-4o_max_loop_5_v200_old_1-24a0719b_tokens',
 'self_rag_k_3_model_gpt-4o_v_304-1cb5682e_tokens',
 'seal_rag_k_3_model_gpt-4o_max_loop_5_v200_old_1-24a0719b_total_cost',
 'self_rag_k_3_model_gpt-4o_v_304-1cb5682e_total_cost',
 'seal_rag_k_3_model_gpt-4o_max_loop_5_v200_old_1-24a0719b_recall',
 'self_rag_k_3_model_gpt-4o_v_304-1cb5682e_recal

In [37]:
# Run complete analysis
results = analyze_rag_experiments_complete(df)

# Access the summary table
summary_table = results['summary_table']
print(summary_table)

# Access detailed results
detailed_results = results['detailed_results']

# Save summary to CSV
#summary_table.to_csv('rag_comparison_summary.csv', index=False)

# Filter for significant results only
significant_only = summary_table[summary_table['Is_Significant'] == True]
print("\nSignificant results only:")
print(significant_only)

# Get specific comparison
seal_vs_crag_accuracy = summary_table[
    (summary_table['Comparison_Method'] == 'CRAG_RAG') & 
    (summary_table['Metric'] == 'Accuracy')
]

🚀 COMPREHENSIVE RAG METHOD COMPARISON
📊 BASE METHOD: SEAL RAG
🔍 COLUMN DETECTION RESULTS

📊 SEAL RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → seal_rag_k_3_model_gpt-4o_max_loop_5_v200_old_1-24a0719b_final_answer_correct
   ✅ precision            → seal_rag_k_3_model_gpt-4o_max_loop_5_v200_old_1-24a0719b_precision
   ✅ recall               → seal_rag_k_3_model_gpt-4o_max_loop_5_v200_old_1-24a0719b_recall
   ✅ f1                   → seal_rag_k_3_model_gpt-4o_max_loop_5_v200_old_1-24a0719b_f1

📊 SELF_RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → self_rag_k_3_model_gpt-4o_v_304-1cb5682e_final_answer_correct
   ✅ precision            → self_rag_k_3_model_gpt-4o_v_304-1cb5682e_precision
   ✅ recall               → self_rag_k_3_model_gpt-4o_v_304-1cb5682e_recall
   ✅ f1                   → self_rag_k_3_model_gpt-4o_v_304-1cb5682e_f1

📊 CRAG_RAG:
------------

## k = 3 | gpt 4.1

In [ ]:
df = pd.read_csv("files/experiment_comarison/k_3/seal_k_3_model_4_1.csv")
#df = df.iloc[:, 30:]
df

,id,inputs,reference_outputs,basic_rag_k_3_model_gpt-4.1_v_307-85bf25a2_outputs,self_rag_k_3_model_gpt-4.1_v_309-7011eafe_outputs,basic_rag_k_3_model_gpt-4.1_v_307-85bf25a2_run,self_rag_k_3_model_gpt-4.1_v_309-7011eafe_run,basic_rag_k_3_model_gpt-4.1_v_307-85bf25a2_status,self_rag_k_3_model_gpt-4.1_v_309-7011eafe_status,basic_rag_k_3_model_gpt-4.1_v_307-85bf25a2_error,...,crag_rag_k_3_model_gpt-4.1_v_308-899087b5_precision,seal_rag_k_3_model_gpt-4.1_v_305-8cb8c3df_precision,crag_rag_k_3_model_gpt-4.1_v_308-899087b5_f1,seal_rag_k_3_model_gpt-4.1_v_305-8cb8c3df_f1,crag_rag_k_3_model_gpt-4.1_v_308-899087b5_wrapper,seal_rag_k_3_model_gpt-4.1_v_305-8cb8c3df_wrapper,crag_rag_k_3_model_gpt-4.1_v_308-899087b5_recall,seal_rag_k_3_model_gpt-4.1_v_305-8cb8c3df_recall,crag_rag_k_3_model_gpt-4.1_v_308-899087b5_final_answer_correct,seal_rag_k_3_model_gpt-4.1_v_305-8cb8c3df_final_answer_correct
0,0004211d-fdd0-46a4-ba01-3a5d916924e9,"{""context"": ""{'title': ['Clayton Oliver', 'Ste...","{""answer"": ""1.95 m""}","{""response"": ""I don't know."", ""retrieved_docs_...","{""response"": ""I don't know."", ""retrieved_docs_...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.0000,1.0,0.0,1.0000,NaN,NaN,0.0,1.0,0,1.0
1,009df37c-3a8f-4a00-b8ed-342f7754257e,"{""context"": ""{'title': [\""2012\u201313 VCU Ram...","{""answer"": ""1838""}","{""response"": ""The context does not provide the...","{""response"": ""1968"", ""retrieved_docs_list"": [""...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.0000,1.0,0.0,0.6667,NaN,NaN,0.0,0.5,1,0.0
2,00e0d7de-631c-47fc-b651-3e69c2029606,"{""context"": ""{'title': ['All I Wanna Do Is Mak...","{""answer"": ""2000""}","{""response"": ""March 14, 2000"", ""retrieved_docs...","{""response"": ""March 14, 2000"", ""retrieved_docs...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.0000,0.5,0.0,0.5000,NaN,NaN,0.0,0.5,1,1.0
3,0169d92a-754a-407c-afd8-5372b4b493a5,"{""context"": ""{'title': ['Alle tiders kupp', 'V...","{""answer"": ""Fomento Econ\u00f3mico Mexicano""}","{""response"": ""Coca-Cola FEMSA"", ""retrieved_doc...","{""response"": ""Coca-Cola FEMSA"", ""retrieved_doc...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.6667,0.5,0.8,0.5000,NaN,NaN,1.0,0.5,0,0.0
4,016dfba5-d311-462a-8ce5-c9db85d2159e,"{""context"": ""{'title': ['Baidu Baike', 'Social...","{""answer"": ""Wikimedia Foundation""}","{""response"": ""Pan Haidong"", ""retrieved_docs_li...","{""response"": ""Pan Haidong"", ""retrieved_docs_li...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.3333,1.0,0.4,0.6667,NaN,NaN,0.5,0.5,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,fdf4d662-d51a-4c36-a798-e5f9bc5bcc38,"{""context"": ""{'title': ['Archdeaconry of St An...","{""answer"": ""87""}","{""response"": ""Eighty-eight parish churches."", ...","{""response"": ""Eighty-eight parish churches"", ""...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.0000,0.5,0.0,0.5000,NaN,NaN,0.0,0.5,0,0.0
996,fe72dc28-9b5f-41a7-ab23-bdb8e2ecaa97,"{""context"": ""{'title': ['Tralee Speedway', 'Di...","{""answer"": ""Dirt track racing""}","{""response"": ""Dirt track racing"", ""retrieved_d...","{""response"": ""Dirt track racing"", ""retrieved_d...","{""inputs"": {""inputs"": {""context"": ""{'title': [...","{""inputs"": {""inputs"": {""context"": ""{'title': [...",success,success,NaN,...,0.3333,0.5,0.4,0.5000,NaN,NaN,0.5,0.5,1,1.0
997,feab479e-056d-4066-9cf

In [42]:
df.columns.to_list()

['id',
 'inputs',
 'reference_outputs',
 'basic_rag_k_3_model_gpt-4.1_v_307-85bf25a2_outputs',
 'self_rag_k_3_model_gpt-4.1_v_309-7011eafe_outputs',
 'basic_rag_k_3_model_gpt-4.1_v_307-85bf25a2_run',
 'self_rag_k_3_model_gpt-4.1_v_309-7011eafe_run',
 'basic_rag_k_3_model_gpt-4.1_v_307-85bf25a2_status',
 'self_rag_k_3_model_gpt-4.1_v_309-7011eafe_status',
 'basic_rag_k_3_model_gpt-4.1_v_307-85bf25a2_error',
 'self_rag_k_3_model_gpt-4.1_v_309-7011eafe_error',
 'basic_rag_k_3_model_gpt-4.1_v_307-85bf25a2_latency',
 'self_rag_k_3_model_gpt-4.1_v_309-7011eafe_latency',
 'basic_rag_k_3_model_gpt-4.1_v_307-85bf25a2_tokens',
 'self_rag_k_3_model_gpt-4.1_v_309-7011eafe_tokens',
 'basic_rag_k_3_model_gpt-4.1_v_307-85bf25a2_total_cost',
 'self_rag_k_3_model_gpt-4.1_v_309-7011eafe_total_cost',
 'basic_rag_k_3_model_gpt-4.1_v_307-85bf25a2_recall',
 'self_rag_k_3_model_gpt-4.1_v_309-7011eafe_recall',
 'basic_rag_k_3_model_gpt-4.1_v_307-85bf25a2_precision',
 'self_rag_k_3_model_gpt-4.1_v_309-7011eafe

In [43]:
# Run complete analysis
results = analyze_rag_experiments_complete(df)

# Access the summary table
summary_table = results['summary_table']
print(summary_table)

# Access detailed results
detailed_results = results['detailed_results']

# Save summary to CSV
#summary_table.to_csv('rag_comparison_summary.csv', index=False)

# Filter for significant results only
significant_only = summary_table[summary_table['Is_Significant'] == True]
print("\nSignificant results only:")
print(significant_only)

# Get specific comparison
seal_vs_crag_accuracy = summary_table[
    (summary_table['Comparison_Method'] == 'CRAG_RAG') & 
    (summary_table['Metric'] == 'Accuracy')
]

🚀 COMPREHENSIVE RAG METHOD COMPARISON
📊 BASE METHOD: SEAL RAG
🔍 COLUMN DETECTION RESULTS

📊 SEAL RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → seal_rag_k_3_model_gpt-4.1_v_305-8cb8c3df_final_answer_correct
   ✅ precision            → seal_rag_k_3_model_gpt-4.1_v_305-8cb8c3df_precision
   ✅ recall               → seal_rag_k_3_model_gpt-4.1_v_305-8cb8c3df_recall
   ✅ f1                   → seal_rag_k_3_model_gpt-4.1_v_305-8cb8c3df_f1

📊 SELF_RAG:
--------------------------------------------------------------------------------
   ✅ final_answer_correct → self_rag_k_3_model_gpt-4.1_v_309-7011eafe_final_answer_correct
   ✅ precision            → self_rag_k_3_model_gpt-4.1_v_309-7011eafe_precision
   ✅ recall               → self_rag_k_3_model_gpt-4.1_v_309-7011eafe_recall
   ✅ f1                   → self_rag_k_3_model_gpt-4.1_v_309-7011eafe_f1

📊 CRAG_RAG:
--------------------------------------------------------------------